# Qwen from original pretrained weights: augmented MCQ training

Start from Qwen/Qwen2.5-Coder-7B-Instruct at the pinned revision with fresh LoRA
and a fresh optimizer. No previously trained adapter is copied or loaded.
This is fine-tuning pretrained Qwen, not random initialization of the 7B model.

One epoch, 5,151 MCQs from 1,717 training sources, one deterministic shuffled
option order each. Three saved checkpoints with validation loss and accuracy.
Validation192/test269 unchanged. Rank16/alpha32/dropout.05, LR1e-4,
NF4/fp16, batch1/accumulation16, cosine/5% warmup, max4096, seed42.

Attach your EXISTING private `mcq-1ep-3scores-1117-data` dataset. The notebook
embeds the updated trainer and never loads the dataset's historical adapter.
GPU smoke and the full run initialize independently from the pretrained model.
Results are under `fresh_training/`; final accuracy is printed in the notebook.
Historical control comparisons are descriptive, not matched-budget ablations.


In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['USE_TF'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
%pip install -q 'transformers==4.57.3' 'peft==0.17.1' 'accelerate==1.15.0' 'bitsandbytes==0.48.2' sentencepiece matplotlib


In [ ]:
import hashlib, json, shutil, subprocess, sys, zipfile
from pathlib import Path
from datetime import datetime, timezone
WORK = Path('/kaggle/working')
os.chdir(WORK)
EXPECTED = {'augmentation_manifest.json': 'fdfc5970b0ebf470c49a1b66dab91a755bb115f1766eb1f5691bb4a18109b8c3', 'control_results.json': '3d1776bcb534adcb133dcc8ec3f888dd3465a3a3d60c296f203bef4f977ea6cd', 'control_test_predictions.csv': '2713119534212cf3fdea1941c002dcc108056cc718913fa8eed97dc43bdd034e', 'control_test_summary.json': '2783dc5c91f312ac9b509a51c2a466361664fc805fc411e9868c751af9d5b836', 'control_validation_rotated_predictions.csv': '4773c52facd3877306180d98d1cbdbb36c0706d9c5ce1c4204e049ab9695ca0e', 'control_validation_rotated_summary.json': '73007145bf4b619bda78e7974c637f4780826f87219e2c56c1629849acddfcc8', 'generation_provenance/acceptance_policy.json': 'df0c1d9f7b16e98cd135b4616e42dfadd07010846378d39e686b5f319c3c12ac', 'generation_provenance/approved_five_fixes_receipt.json': 'baaed911806bdd26ea341bbf852d7390db979617162c46993768c1341fae6fc2', 'generation_provenance/approved_five_fixes_verification.json': '5eed1602492246ea53c9eb14b7e1a5a4b72a2831f4fc6feb35804b3e46bbada0', 'generation_provenance/audit_progress.json': 'c5be7e560d145aaf78b4e848e8eb900a578c9533a60bb7c1ed346eac370a14df', 'generation_provenance/audit_progress_console.json': '3b6b4d4402f11a71706a4a021390579dbb6bc3148b587cba2253f5c8d1c6735f', 'generation_provenance/IMPLEMENTATION_STATUS.md': '1908db45cf73c84be62ba6f0ea5df078caf30909a8c6e31d0a580d4031ad555b', 'generation_provenance/manifest.json': '9bc0fdece5c7ef49dc050bde23d397691ac2a85fd5595beca6a31e42a3e94502', 'generation_provenance/merge_console.json': '7e30a87b1afa9bb8fbc0f054f237472bd9aaabf5745d3b093d01f1c8763543b8', 'generation_provenance/original_repairs.json': 'a795c29de7492e7eed5e3fbc93cd681ef873e3c29e2f02203ef77cb1128c261c', 'generation_provenance/original_review_notes.jsonl': '5b43008345e12ee24c1ff11a518b882db13e29a75f7ddad4155b6ad1b55f3b51', 'generation_provenance/preflight-single-random-v1/length_exclusions.json': 'a5338d955b09046ec0b16f3a9625b7955c763aae07dc722e474e6078745f932f', 'generation_provenance/preflight-single-random-v1/metadata.json': '04372eb5ea5991d15ad9b15ae9233387844ee8c955126ae8405d2291128de2d7', 'generation_provenance/preflight-single-random-v1/rotation_map.jsonl': '0f9800151b495c9ecb707f09b3905af9ccde514cc6e80cddd1d2a764708cba3f', 'generation_provenance/primary-decisions-0000.json': '833608c8138742a9dac14baaade821fdc10b115f643b692795ed7abfc316b56b', 'generation_provenance/primary-decisions-0001.json': '830b20efd430c39cb629ebd563bcc08826f0825433bd8ec3ba5818ed9a6a69a3', 'generation_provenance/primary-decisions-0002.json': '1e8710dc98f0ebe0b3c202ecfb80c826bc8ece427fb93e44a5e7f703da5d8994', 'generation_provenance/primary-decisions-0003.json': 'f410477e836ff0e7f504d2695679ad0accb15354c6d864815b5a64ba9b0284e4', 'generation_provenance/primary-decisions-0004.json': '51deb96d8d218455b2bacd252b1742d9efc523084e9fc1b9efe030723fb3178e', 'generation_provenance/primary-decisions-0005.json': '71aefe3ecf04b3fe15ec2ede86b3c1941d30c70f0ee11ab13331fdf4e965b227', 'generation_provenance/primary-decisions-0006.json': 'e9e25c5bba10afb9cfe36beae6f28bba3a7233279722861ffe02e27bb1c24f39', 'generation_provenance/primary-repair-decisions-0005.json': 'b474305e2d5e01db43b507b8d5b2634269b6996d085334b59c7ea9cc30196e5e', 'generation_provenance/primary-repair-decisions-0006.json': '0069766171dfe5510755d213a009268fd4b5f8ce9d77f4d7c9533a18c4f40242', 'generation_provenance/primary-review-snapshots/0005.json': '09f549a1cba9e99e58f145caf7626e4d8020d055551b135ea2d3ba6cd22ff659', 'generation_provenance/primary-review-snapshots/0006.json': '46e47b109fedf487ac11e63fb446e7c624f6b790a7b16f600b6131a175439f1e', 'generation_provenance/primary-review-snapshots/0006b.json': 'ce2bd97313a6d91c0eacf18f65dbbcef9b7181056d62b6df1151a05649685647', 'generation_provenance/primary_candidate_repairs.jsonl': '6c0ccc84ab5c8c525aab9aa7a442035e4a79093df961fa0adc0ef97ec36a3923', 'generation_provenance/primary_reviews.jsonl': '36965c3d5c3568ee5aad0a14837edb4cf843455b15124a362c4303490bb35a0c', 'generation_provenance/proposed_original_repairs.json': '6f4e201f2390e5cf0eb89298aaa75883ebc6e746957534458f84a7720014fa3d', 'generation_provenance/proposed_question_repairs.json': 'abd9c6cc076485d4197081c6c1d4c41021bca978a35abfad0d9ffea8afe2b924', 'generation_provenance/repair-history/pre-five-approved-fixes/original_repairs.json': 'ca2b6f83faec696d53c237b997a5d84dc017d9c207fd17a5b375559e64f6a407', 'generation_provenance/repair-history/pre-five-approved-fixes/proposed_original_repairs.json': '080d757f86d497442c6e3c76f674d677b73ea173843c2792ed78e5d88bb21992', 'generation_provenance/repair-history/pre-five-approved-fixes/proposed_question_repairs.json': 'c3573a4a17901926303aae3f65f5e688716e2b3819537e252d583fb13459a5ab', 'generation_provenance/repair-history/pre-five-approved-fixes/worker-1-contradictions.jsonl': '639e74df6f74825bc7d3fc973a9e1ee107abfebfd50d4f8e1d6c277dc3683d99', 'generation_provenance/repair-history/pre-five-approved-fixes/worker-1-source_audit.jsonl': '0e5a8295c286669529e12768c54c7354ccfd4d29186e65684202c9fd2714b592', 'generation_provenance/REPAIR_DECISIONS_NEEDED.md': 'ae7a4fabd995bfeed523de256afdbb1949355b013b47b2aa94c4ecbfec2e1ef9', 'generation_provenance/training_protocol.json': 'd3ca0b9e5e05cd7df5f5ac64e87b845654544046f55d80babf85f345cffbc0f0', 'generation_provenance/worker-0/batch-0000.jsonl': '937001fb45a0f0465320d9eb529a281fa619ebde18078ad6ae5c9cc76f8d1ddb', 'generation_provenance/worker-0/batch-0001.jsonl': '4c4a53a6b311c01cf6e36e3954e779dc50f78063c00b0b9cded0cc2946fad673', 'generation_provenance/worker-0/batch-0002.jsonl': '0b4887099a1338875a3477ddbff10d1090bc3af3ab5cf5562a14182179c68e1b', 'generation_provenance/worker-0/batch-0003.jsonl': '11a84be17995290dccbaafd60b32f20fcc25705f6e3def4440adb591e496e76c', 'generation_provenance/worker-0/batch-0004.jsonl': '1203691b2ed3132c8c8725ef2d16c3d94231159dec4c13292c440aef910f11c4', 'generation_provenance/worker-0/batch-0005.jsonl': '6467b200400d620f5604142538b3368c1b1b9f5f260cc3a6d7846baa4b3285e4', 'generation_provenance/worker-0/batch-0006.jsonl': 'e5bb1d58b95f7dedb738169a2330b5e2aeb43f0c0f9aaf37d6aba3f73d779e70', 'generation_provenance/worker-0/batch-0007.jsonl': '9d33f0966adc8a946139c74c4fbd09f31660c9494c4c253f3d93ab4f6da3a743', 'generation_provenance/worker-0/batch-0008.jsonl': '996c9876beb72f778de5fd88fd338fb876f6f45b6c0f3cea03353ecdff98c83c', 'generation_provenance/worker-0/batch-0009.jsonl': 'c338246797b0a12d80fac9c133f2d32ccda972060f2fe2ef4b0955e6436eabd2', 'generation_provenance/worker-0/batch-0010.jsonl': 'caa3a2e854c3e0355945a9efd644a153d3a804cf83c2b2fff1d649f920f50bde', 'generation_provenance/worker-0/batch-0011.jsonl': '7d82b75e78343e91ca0ee89c0c0f2a6ba3a952c7ee817ea5d670eeed7f0938d3', 'generation_provenance/worker-0/batch-0012.jsonl': '2d3e9e3c4ee9d3cdc6ffe624e3b46d65ab118c31f5c319047ede5926524b6c28', 'generation_provenance/worker-0/batch-0013.jsonl': '0409516f7f1278c4849dda5b65d7c2c36d364015ccbaac646ab917d77d57f139', 'generation_provenance/worker-0/batch-0014.jsonl': 'f00b93e5aafeb3be42774e85f7de788886d6e2258ad7fd7fdc28cb8897d491e9', 'generation_provenance/worker-0/batch-0015.jsonl': '8415e74b33016e880d50df1d181818cb9315384ed5bb839ad43104748fc5cbe4', 'generation_provenance/worker-0/batch-0016.jsonl': '19344f2eb3fb6fe58c091312deee968ffe28bdc64c2dad760a07309da0648701', 'generation_provenance/worker-0/batch-0017.jsonl': '4ce9b40d3b80482e761980a75a03968eb41738625cb8eeb243b66accfcda4443', 'generation_provenance/worker-0/batch-0018.jsonl': '1399c439b3d1f7ee441ad5a396bb22925239811f86e0ea2fe5258afae3e58b4e', 'generation_provenance/worker-0/batch-0019.jsonl': '1c2f140ee7bcc2bb02eaf9ef043bf42645b0f8b556c8399c94d3b0e7c4fc4216', 'generation_provenance/worker-0/batch-0020.jsonl': '4ec7195cb1636977cf8750d14713bcf4698cf81de8b0044f280997f6e6ee144a', 'generation_provenance/worker-0/batch-0021.jsonl': 'e6b7470abdf7a80805798ec9a58e1a38cecfda1c6192dc6f5a9dcc3c50fa5380', 'generation_provenance/worker-0/batch-0022.jsonl': '66840fcff069ff3f5026353516f4cd29f87fe224a478fd6893ef64927dd91802', 'generation_provenance/worker-0/batch-0023.jsonl': 'dfb0c063063de566366c8782508d2621b9816a7d1a2e4fd8d38979f639dd8152', 'generation_provenance/worker-0/batch-0024.jsonl': '0306939ac6ffe8ecd0c976b82c5e7df0d9883eb495d10b9e48535fa5abead967', 'generation_provenance/worker-0/batch-0025.jsonl': '09b2afbc34be2df710ee4c834937da9e6da4ff094dbea0c32a7b059214718b62', 'generation_provenance/worker-0/batch-0026.jsonl': 'd17007c6aa5c6ce5a7612a3c85e14111f55c347cbbe2b50b6e5c8286ba41d542', 'generation_provenance/worker-0/batch-0027.jsonl': '1dc99580225479e5740d11f610c9b91e74306d69e7eeeb12936231037a4ea1e1', 'generation_provenance/worker-0/batch-0028.jsonl': '871bd61821f966379ae99be21cb229a55f2d705062d1a64e69934f9818c08128', 'generation_provenance/worker-0/batch-0029.jsonl': '9fc2a8cfd6d2f0a8ba721e2b8f1d9ac0a43894b670226a91f6f6f799922d3d4e', 'generation_provenance/worker-0/batch-0030.jsonl': '2310529c41f140a0b3d444685b48b482c3d08dab4a6a12905e513b4e6e92a276', 'generation_provenance/worker-0/batch-0031.jsonl': '77685d3651a55efd821ac142a9b1b51bafed1089aa9a6990fb7a5b6b7c31d462', 'generation_provenance/worker-0/batch-0032.jsonl': '850191427e321d15d549a1dc94bb905760032dcb33f18aedf5c3bf5cfb2dc9f1', 'generation_provenance/worker-0/batch-0033.jsonl': '86588f79979a7947c6818914ac90b002513a9217b752c4aa0c741d64da0b744b', 'generation_provenance/worker-0/batch-0034.jsonl': '806a04b954bee3e6e30b2efe5485c8cf3e4d2b6886d57cc4873f4ebaef4eac2f', 'generation_provenance/worker-0/batch-0035.jsonl': '870c0c9dfffec3e50318c01da19a730fce0a0f2267eff14d564737b360d74325', 'generation_provenance/worker-0/batch-0036.jsonl': 'e0b6aad135e38eced4b621a77061651f839ea2ed494c07d3c1453fb9f8fec877', 'generation_provenance/worker-0/batch-0037.jsonl': '687e7d42c36d7e25f6764468f0ef65f582324224058e8ca0a5d9ff925c961fb3', 'generation_provenance/worker-0/batch-0038.jsonl': '864101ac3a50e22e3b1e47080d42434815c53386f00af8498c358c69ccdadd05', 'generation_provenance/worker-0/batch-0039.jsonl': 'fb20760f2a2b78d9704f7b0a8002489bb7e924591796e95dfeb25d2de9379d56', 'generation_provenance/worker-0/batch-0040.jsonl': '238cde0163abd38c81e1f5b13837ab5fefd11fab7460f50f5ddf664ee0d37f7f', 'generation_provenance/worker-0/batch-0041.jsonl': '273bb2a3b78b464b940775e285bf98075c0a3474156b87bc45df959784b105a8', 'generation_provenance/worker-0/batch-0042.jsonl': 'ee3424cc3773e4576cab566e0a38f06caf5a345782cd3c4199c689e03195bf2d', 'generation_provenance/worker-0/batch-0043.jsonl': '72d857b7d2dd0ad6033eda28da03fbd2daed9ae8eb2a0fc2a6442945935f280d', 'generation_provenance/worker-0/batch-0044.jsonl': '2b661f19fde8c7a60ee5006944e63ded8f60184b22107b9aad6c7f15f967daf1', 'generation_provenance/worker-0/batch-0045.jsonl': '68137767b8597a15583f236cf7ea5a44c4d78ff3dd976167d65efa6f6dbd9e2f', 'generation_provenance/worker-0/batch-0046.jsonl': '8800e67fc574c2aeb61a1822e3ecd6c57eedf997fa6c18de7bab6b502f594de8', 'generation_provenance/worker-0/batch-0047.jsonl': '2070a116ad822de37678abbb2c14ba684c6206af9e1004af52d7a14359aa5e3a', 'generation_provenance/worker-0/batch-0048.jsonl': 'a326863283dfae9e72c2e29a52980d12ae7183411ae715a78942462328356af4', 'generation_provenance/worker-0/batch-0049.jsonl': '57c0ef8cf20c9ae7d4b322867fbc34f211eae96ddf489e8f8adff29479768a15', 'generation_provenance/worker-0/batch-0050.jsonl': '40a2e2f88ad5707fb7a7b2a6452d02c31182f69945aed3d4d9026445de18fc1a', 'generation_provenance/worker-0/batch-0051.jsonl': '389619b812e7bb0de344a7d3d20b5f7d57b38f9dbf4611a9939ff7ec9f65595b', 'generation_provenance/worker-0/batch-0052.jsonl': '686da0fd73c040884f3a41c11ffde7ceec616bea3ca36337db86c6b7a38f6cff', 'generation_provenance/worker-0/batch-0053.jsonl': 'a58df8cdad7b901dd2cdc93907c04a061e56bcd72ba9a16b777ed4b5d2294f6f', 'generation_provenance/worker-0/batch-0054.jsonl': 'f02633aff95b98cd50f39623e92080ba0ce3eea3ada5cecf9d5d5a3a98baa2a6', 'generation_provenance/worker-0/batch-0055.jsonl': 'ce458c4d401ff970d3e9d274a98df2a93e6ab53a5bf45ac4619fec750c6f8278', 'generation_provenance/worker-0/batch-0056.jsonl': '340950cd2cef55668333807eebb5c20834e7fc3072081279d731f3d40ab3c331', 'generation_provenance/worker-0/batch-0057.jsonl': 'b5ccd5ac3928b7601f070e429a3101868a83fbd926554e4ddc2140c74a6f0a94', 'generation_provenance/worker-0/batch-0058.jsonl': '146a350b597fe8e9c1ab844917150d6508452c123a9cdd21128c573189751d74', 'generation_provenance/worker-0/batch-0059.jsonl': '43e05fdd9f44f7dc691296647037091b77b43a7597d0f74ac96ab501e8bd99b4', 'generation_provenance/worker-0/batch-0060.jsonl': '3445c6ca18a4925555686270b793056caece5c2462461e5f4f5e5fbf2b386dcb', 'generation_provenance/worker-0/batch-0061.jsonl': '28a017c73f616b93f4396ceb5dcd17f3a77be93c48a49bf74b1b873ed65e3536', 'generation_provenance/worker-0/batch-0062.jsonl': '0735ce7c8783642ea183519b5830d03014475c7408d98e86afd5b0d5fca0bffd', 'generation_provenance/worker-0/batch-0063.jsonl': 'ee009a027727eef757dfccbbc89596c232bc3e5ad58b205c1e838e00ecae7c78', 'generation_provenance/worker-0/batch-0064.jsonl': '6e040bd8fe49b8d5ea4f5955d2818edf6c90d74d9b6dbe09f56566b284796d71', 'generation_provenance/worker-0/batch-0065.jsonl': '5b8cb5b257d14ed31c30a6a8bd7891631ccd3eadfcf8b3f2f43c02b059778989', 'generation_provenance/worker-0/batch-0066.jsonl': '5409f22b4c38a8b748047d1f22d622b0b27acfff36b7bfc1d744a778c53fe9ca', 'generation_provenance/worker-0/batch-0067.jsonl': '632a41470cb4ca96d338fb80fb9914a332b383f2fb9617e074fa4a5b9afb99f4', 'generation_provenance/worker-0/batch-0068.jsonl': '6eb52862f101776e9e378b2c6cf4d7395ebdb863bff88784fb1de792dc5ec07e', 'generation_provenance/worker-0/batch-0069.jsonl': '1a73a8c3569e2820df605b7db6ad2bfa2b8ee1fffb8aa87dd2593f930578fa7e', 'generation_provenance/worker-0/batch-0070.jsonl': '70a1d6d79f652810ba79231efae0e749cc0d12b550f2aaab0e879701eb647e02', 'generation_provenance/worker-0/batch-0071.jsonl': 'e849daf279aac5cea3b00b8b642f8d9570d546e82a751534ac8b26fe7f631e6b', 'generation_provenance/worker-0/contradictions.jsonl': '60061c8f2b18ebba1e5203b6740719f0402cf8bec1a02ebc4efe4d03f01f4a70', 'generation_provenance/worker-0/evidence.jsonl': '1b12777bb51432e7bad65c141357929e78deb67c8d452925175e28ff3ca029e2', 'generation_provenance/worker-0/progress.json': 'b044107ba4e65da553adb01bc917c6cd66198944779322163675dc419ff02e85', 'generation_provenance/worker-0/rejected-drafts/batch-0000.initial.jsonl': '092692b3e25a845d7b4a39b304dcc8d1e59247b34b9b6a11795a9a59f159da26', 'generation_provenance/worker-0/rejected-drafts/primary-before-primary-decisions-0005-batch-0000.jsonl': '45504e1f49fad3d72f70b8b5f9fc586b22fa4e2610f7c2de89e8949b8ffb6a57', 'generation_provenance/worker-0/rejected-drafts/primary-before-primary-decisions-0005-batch-0001.jsonl': 'e18ce59034e89c0cd172ceefb167c90172b40c645771c9ea8e2b2db6ef49f824', 'generation_provenance/worker-0/rejected-drafts/primary-before-primary-decisions-0006-batch-0001.jsonl': '29ef1344de179f501c9823db49c57375eed413a07b17f15aa704dfd1c58cc137', 'generation_provenance/worker-0/source_audit.jsonl': 'aa9fcfaca2b00cbaa6a26b920184190b6bee866b951b1b771abe66b9c75715d0', 'generation_provenance/worker-1/batch-0000.jsonl': 'fc8e620265a5e7ac54ce3d700b624bbf653c0380bb3d172e9d9f6267acaf156f', 'generation_provenance/worker-1/batch-0001.jsonl': '5ad10488ee0e6021c62d132a79cf42b4d4358ae124d5e88f6ab7c9ed4dde0fff', 'generation_provenance/worker-1/batch-0002.jsonl': '864d07cdc087dcfab165f74310e8c4664866049c0a7c73a00b49bf61fd15a0a5', 'generation_provenance/worker-1/batch-0003.jsonl': '377c21b95fd77c1c4c2129017ca94b1a1a547674faf4cd1a3cd56a2e07a51f8d', 'generation_provenance/worker-1/batch-0004.jsonl': 'b300dc95507584ddc5b070da79bc912864a33fa25c0a0cadf8f1faa3068c7180', 'generation_provenance/worker-1/batch-0005.jsonl': 'a77ca2be31654a5e595634fc11fb6c802eeaeb5ffe81dc9c8b242409ced09466', 'generation_provenance/worker-1/batch-0006.jsonl': '2aed333e274cd24429b216fe8ef819910da08d28b9ec91c7395b49d9fd9ec599', 'generation_provenance/worker-1/batch-0007.jsonl': '3ce07f265a11ef8c533ee00c7ef2c3f0518588ee21a3a3da571e08399aa2e2ef', 'generation_provenance/worker-1/batch-0008.jsonl': '0760e254bead757ad6e4fe6088161942bb8239d6812298d28e5ccd4fb996790c', 'generation_provenance/worker-1/batch-0009.jsonl': '8e4270fe6996420434435da739bfad7db22a8ce3410fd42ea6c64cb976bf45c3', 'generation_provenance/worker-1/batch-0010.jsonl': 'd5af3082acb769a59fc52437edf3eabacb67718ef8e5e7348bb5c3204f2c91f9', 'generation_provenance/worker-1/batch-0011.jsonl': '4acc2cac0e4012f2fcd91cf23820763c4897e03307d12a858686ca5e364d4229', 'generation_provenance/worker-1/batch-0012.jsonl': '062b7eeb8ddffaf0d1f01aa2727109ef8b275c9906d5aef53d46d6048fbd102c', 'generation_provenance/worker-1/batch-0013.jsonl': '874911fdc5202c7d1e4891b16a5f55519d5d435bfbef1f76d56188f941c35db0', 'generation_provenance/worker-1/batch-0014.jsonl': '9fa8d3c990fb18ff60949baded1fef7b301f2ab258b7d843d6653d32cc6b2c87', 'generation_provenance/worker-1/batch-0015.jsonl': 'b5e49c925fc0343c73a5f8647cc34c712f6b847bdf66a8920b061999a9a1fcf0', 'generation_provenance/worker-1/batch-0016.jsonl': '409dd2b447372587e7c0ff7746f3ce7afee1b632cf03adf079969d6f98715004', 'generation_provenance/worker-1/batch-0017.jsonl': '976faf6492ba8c79cb8fe1a8d56511a31f0dc625413ae452d357bd0150ebbdb8', 'generation_provenance/worker-1/batch-0018.jsonl': '57a15692d721cce55721dca263ba78c2e15ed775a5bc49ba6d0bd7dad9e4f4c8', 'generation_provenance/worker-1/batch-0019.jsonl': 'bc73925c011fe90552892f1c3fc9ec4179745e97316b56bf4dd4ca6cb2a3219b', 'generation_provenance/worker-1/batch-0020.jsonl': 'e3067467f65ace5962cdbf13a54f2befb8c22a450900318f4d77a4fad89fea1a', 'generation_provenance/worker-1/batch-0021.jsonl': 'edf9e331b1feff52711f9a7180a83b6d2ab2bf59b1b99024a29b792a74bc693e', 'generation_provenance/worker-1/batch-0022.jsonl': '2b694b3021fab48f020daa704d7149de45f0c4e44af65532e031b1f7777ecb8f', 'generation_provenance/worker-1/batch-0023.jsonl': '45e3ee4862fbd617731cac36878e0be34b1060c6d8f779dc6cec6741eb17a776', 'generation_provenance/worker-1/batch-0024.jsonl': '4da3803c9cd4a2c971457ceb12d4097fca3715060dddf18325b3827cbcea377c', 'generation_provenance/worker-1/batch-0025.jsonl': 'c19157d6a22b4cb1d92d7c9b42d39c2967a75958f93216fe46b3dc4ee633ec31', 'generation_provenance/worker-1/batch-0026.jsonl': 'e457a81c05022e063fb6de22ca46f9b01d3ecd2abe3c4e0623ff613a52e18448', 'generation_provenance/worker-1/batch-0027.jsonl': 'a091703ccd57ccf03de66e2262255d8ee6b90618a0452137c38813f650fca55b', 'generation_provenance/worker-1/batch-0028.jsonl': '9ba07ba07ddd5dcd6298e5ca5bd467376877d8ad9b9031a70a9a097280b77d29', 'generation_provenance/worker-1/batch-0029.jsonl': 'dcfc400f68c8f629196ab3b5b898960cbec81f14df957ae6f6e9dd42aa70c6d6', 'generation_provenance/worker-1/batch-0030.jsonl': '2156ca52ca4e25d9edf39f7b7539dc15310f9d5460d9a854e04a7cb214d15c2a', 'generation_provenance/worker-1/batch-0031.jsonl': 'f7e2ade999cb213c2e1306b6c39ed23d078764d46566767e1d02798cde4e58c5', 'generation_provenance/worker-1/batch-0032.jsonl': '95ea3466d20cd13c64cacf0edd2f0b94545576d1c09efa8aaf3c890887412f1d', 'generation_provenance/worker-1/batch-0033.jsonl': '13c3cacf39eb12548437e5a93ecc875dbec4a9a730119675d51ebd53b1daa602', 'generation_provenance/worker-1/batch-0034.jsonl': '936fc35dfe303e3d9c4356d227cea13890e1ff9628c5c904ba067d9dbbb14e5d', 'generation_provenance/worker-1/batch-0035.jsonl': '03bfec00a455b7f07e709092d44005d8e918b6c9de17c08ca62cf97c65062dd9', 'generation_provenance/worker-1/batch-0036.jsonl': 'b2c2f0a0bc707ab94a5d47d27e2a3125920fa7e6441543f71bec3d5f180fe3d3', 'generation_provenance/worker-1/batch-0037.jsonl': '2ce14dc82c89b0ddd9ae156596e3b8803a7e2e70a887ddc26f3759e87ca1d1da', 'generation_provenance/worker-1/batch-0038.jsonl': '90bd917f8bd30162244efcb8698d929b1355f6d725ff8fc66056cceae9755270', 'generation_provenance/worker-1/batch-0039.jsonl': '492ef37ba5c7b5a24b502362f2a3af9c7f8f3e83585aadff0d82e455447f549a', 'generation_provenance/worker-1/batch-0040.jsonl': '278717f6cdb5d23dfa96dd89d793fabf5e9a60fa9e38f1a5f66cb444f9b45813', 'generation_provenance/worker-1/batch-0041.jsonl': '03aa1fabada5078f6767e7524c5b6f5f2fe3c5bfc1a204acf02ec3dc4de04d3e', 'generation_provenance/worker-1/batch-0042.jsonl': '3d38173e6288e7bfd638dbbd6b39a4fd919e98176a1f915a0247337d57494e9f', 'generation_provenance/worker-1/batch-0043.jsonl': '891aabf373dd478575c2fde270d5e5e73d6fd788cde3902cf8cd7a6ff73e931f', 'generation_provenance/worker-1/batch-0044.jsonl': 'd83cc8271e92d525e917682336afb56c41760890302ca8f56f2b987afbea3a42', 'generation_provenance/worker-1/batch-0045.jsonl': 'd049e55b50261eefc8b8b11e6d35da6aba558044c5b965da25ecf95d0735e7ff', 'generation_provenance/worker-1/batch-0046.jsonl': 'd9d2d4403c267cc4cbf18ab854297acf9bd27f83138e8204a250d5b0e61b84ec', 'generation_provenance/worker-1/batch-0047.jsonl': '02e964f352634613a0c3b8db042ada745b82a9f253b76b1d1c9ad0464bb5f1b7', 'generation_provenance/worker-1/batch-0048.jsonl': 'ba28c06b671232ed2c92eca4efa76143a4381a9578aa82352765f22d4424f87a', 'generation_provenance/worker-1/batch-0049.jsonl': 'd6147bb082b12291a8eab8b8421fc2539ffc3ede6b854ff3d36eb4280596724a', 'generation_provenance/worker-1/batch-0050.jsonl': '6a352e770596ca03dbe130d0b3919af639074392fd2be97b25831e8a05eca302', 'generation_provenance/worker-1/batch-0051.jsonl': 'ab0d4e56be55d37a2295639943d7c9d962bdbf5cb7ead3dfe38ed3734af2d643', 'generation_provenance/worker-1/batch-0052.jsonl': 'c6cb4aec6ce81bacdd4eb8763faec52faad537542e5d52cb6ab5d8356417fa56', 'generation_provenance/worker-1/batch-0053.jsonl': 'bfb85735397d55d491a1a55bed7d58bbb0f5d4194812f1a668846c6de59a042e', 'generation_provenance/worker-1/batch-0054.jsonl': '6c0bec98d307437e314f52cb61047f989c4a9c528f307ea980641edce682faca', 'generation_provenance/worker-1/batch-0055.jsonl': '6f0e1833230adcf588989feb560c9bbe1f533db5e28f3a51ac3821815de52620', 'generation_provenance/worker-1/batch-0056.jsonl': '2bc1fae21121986e5345068575ae791154ba1b30e003704ffda6053b5f8a259a', 'generation_provenance/worker-1/batch-0057.jsonl': '6ff6381d89c0c2a20e5afdeb92dbf3adfaacc03469448c60b4421f1435d9ced9', 'generation_provenance/worker-1/batch-0058.jsonl': 'f53fc8f1a7ce9d77e625d8c7bc617872f1840ce2a60c557318a7b184c32314eb', 'generation_provenance/worker-1/batch-0059.jsonl': '87d91284724aeb098fc32880c5b244f13f145f13ec10127af4e4058a5d76456d', 'generation_provenance/worker-1/batch-0060.jsonl': '71adc483b486768a4b4a49ca076426f331badba7aa7d06eee8f076de26aaa293', 'generation_provenance/worker-1/batch-0061.jsonl': 'e853a82bec125b69178dd6ef3bc2caccdaf9f8b385375684f550a451f086b1e1', 'generation_provenance/worker-1/batch-0062.jsonl': '2270976d34f0ac04c757acaad77082f5c10d2b78ae7fa5d74c3907276967011d', 'generation_provenance/worker-1/batch-0063.jsonl': 'b538a7f46c83b6bf759d01b17aaf80a6485137fef7c274637d1265d5c0b9cab9', 'generation_provenance/worker-1/batch-0064.jsonl': '6099f09ebfded236d77e6ce328a4079841fe7e25c277b11e3ff8ffa8c6c24a5f', 'generation_provenance/worker-1/batch-0065.jsonl': '04338fe7284060d73c1077e35bf4e8046eed1e7db8fdf1c11d45fce1ab4a6e32', 'generation_provenance/worker-1/batch-0066.jsonl': '61e69605bf185c54b34137edf260d0c6eb1a59e0d3b8380d6ce8b778a62a0c81', 'generation_provenance/worker-1/batch-0067.jsonl': 'cf52c720326762f6c9dce220fd91a558c8e58f77ac1c8b58e7abfdfbc35892e3', 'generation_provenance/worker-1/batch-0068.jsonl': 'fc5fe3899f0453ad8e0041f1ee8d290fbb08f0e95e63ab7f1bed71b930561887', 'generation_provenance/worker-1/batch-0069.jsonl': 'a20a1a6898e5c95c935886af1ba1e4572575842288994cbae3890898b78dedd9', 'generation_provenance/worker-1/batch-0070.jsonl': '667b4d1c7d8c23d77cffd1f2942554bcbb67e3b16750eb7e173a58996d9c2357', 'generation_provenance/worker-1/batch-0071.jsonl': '4742dc831933874c8099743df5fb127154147723d760467946e7db8d53a618f6', 'generation_provenance/worker-1/batch-0072.jsonl': '572de9430405cdf54f29f04aa6eaf75467bb7032055a592053751515f71ec7f2', 'generation_provenance/worker-1/batch-0073.jsonl': '18737367bd07f9d281c359635d4fd56d510705cf70487d030d76ad50358cef64', 'generation_provenance/worker-1/contradictions.jsonl': '3e95b5c5868e061d018ae3915244405572f6f85e310e46ca5d5fc25210aafca9', 'generation_provenance/worker-1/evidence.jsonl': '7c3f54e179421de36153a36815beaee746d5118247d8ecb23e812b40adffc836', 'generation_provenance/worker-1/progress.json': 'e46af7fc41a7f4f7dcae1f00b0e8209396a9da1c79f6ba0c673d72cbb7344743', 'generation_provenance/worker-1/rejected-drafts/batch-0000.pre-review.jsonl': 'fb531198252727582ca98a105e9db3361fa4d824f7499167f8132aded44bcea2', 'generation_provenance/worker-1/rejected-drafts/batch-0026.source715.pre-audit.jsonl': 'b56564270e125303ade0bc3dc5cb161ecee505c86de550aa23791140adfb39b4', 'generation_provenance/worker-1/rejected-drafts/primary-before-primary-decisions-0005-batch-0000.jsonl': '866eb436b707b38c1624a71fd189fa39bb03d8728b970e17483f744f80856334', 'generation_provenance/worker-1/rejected-drafts/primary-before-primary-decisions-0005-batch-0001.jsonl': '79dbe5120acad138cb453fb2ee2927556c64ca10bf47ee157c4df9cd3f8bc8b2', 'generation_provenance/worker-1/rejected-drafts/primary-before-primary-decisions-0006-batch-0001.jsonl': 'ab2d3f90e0ddeb308e87137ffafa33c7c5e66900dfdcdac46b6d85207bab4eb9', 'generation_provenance/worker-1/source_audit.jsonl': 'ba28c51ff120dbd4bd68e23b85773325920c83321e66d5b109bf14dcc8a7a1aa', 'generation_provenance/worker-2/archive-drafts/batch-0000.jsonl': '7acb10ebaaf48ba076d8ca6676d7ea6795a8cd587a3ab01953d6a033a3f7bb7e', 'generation_provenance/worker-2/archive-drafts/batch-0001.jsonl': 'aae64d26a8497d976033ab7a0020106d147e23aa68c6af81848919fe2f71ece6', 'generation_provenance/worker-2/archive-drafts/batch-0002.jsonl': '4e72f26cecbb2b54993a70c18346bfa1e774b6e2373065cf57f7b08332f98c0b', 'generation_provenance/worker-2/archive-drafts/batch-0003.jsonl': '6f409a9e419e90ea367a16bce7eae6ff2993c8b147ae03ab14b258724b54b91f', 'generation_provenance/worker-2/archive-drafts/batch-0004.jsonl': '777546602a7dfdf69f565590e4741bb3d2e471c3e1804de73ff9ed3f5f968ad8', 'generation_provenance/worker-2/archive-drafts/batch-0005.jsonl': '4d6f2e951742de23b78435ebe5950d52bc4e561a9fb63570f920907b241c3f9e', 'generation_provenance/worker-2/batch-0000.jsonl': 'ce25ca842ab5fca44076229638e2ae16082e77a67494e55569efb8807ec8d518', 'generation_provenance/worker-2/batch-0001.jsonl': 'b6dd40a04e2b65bb651460e07409d7589cb894b68e567ba99f72fec00ab8d0e4', 'generation_provenance/worker-2/batch-0002.jsonl': 'c98c323a559f11c61ed20ff57e9008fe0fabd73e0b816d4123796c997b8fc5da', 'generation_provenance/worker-2/batch-0003.jsonl': '3bc1ed7b1634258b8694ac27dbfda70da32976494e9ec26a88fa39a4c5b40b30', 'generation_provenance/worker-2/batch-0004.jsonl': '28428e93d852e0649961fa1c1b3671d555cd0add6ad9673baa1f579e25709130', 'generation_provenance/worker-2/batch-0005.jsonl': 'ef33c3edd14890adfd36afb9c5e118da11cd34d470d5a9e3e2799263b44cbcd9', 'generation_provenance/worker-2/batch-0006.jsonl': 'dcad6ccdeb403d662cc1d68186fa018712c3cde0dd2a5938a4621ad61002087b', 'generation_provenance/worker-2/batch-0007.jsonl': '2d46b244765676f565c2ad2b94c5b6d51edc644deae3a88f7ac14084d9d9d0da', 'generation_provenance/worker-2/batch-0008.jsonl': 'c2887d4d61ae0d0aab3dfe497aa6301b8e8a0e37d70d02715000502d7bf8be6e', 'generation_provenance/worker-2/batch-0009.jsonl': 'acc45a83a69d194bf374f937580f543017e70a8eced9f777d1f6df60813178ef', 'generation_provenance/worker-2/batch-0010.jsonl': 'd2f23a0eaa3e840f8f99ad5d2adc4a4841a9e7213698ddebf80c52852febab65', 'generation_provenance/worker-2/batch-0011.jsonl': 'ebcc65768b3b18bc44fdb0fda760ecb3f72b1d2782cae98849e0ab4ef9150eb8', 'generation_provenance/worker-2/batch-0012.jsonl': '5600d37a0954233466ac86ed5907319fde66dc9286cd83f51b8350c38d3186c4', 'generation_provenance/worker-2/batch-0013.jsonl': '5cc9cb8e90dbe020b18a66e5d9aed1f764b020a5a563ff590581f9099f85d44a', 'generation_provenance/worker-2/batch-0014.jsonl': 'd327c9f03889e31e8ae82315de235a2a614856699bc9868cfad723e6eda60dd3', 'generation_provenance/worker-2/batch-0015.jsonl': '706785d24defc1c61614d63e1cffe0981de781f74f8476ecb7b7a179913eddcc', 'generation_provenance/worker-2/batch-0016.jsonl': '8ca0fd8ea0e91c7645a1f97c80b7952f84d6f7def30671f021c60102ad2caf51', 'generation_provenance/worker-2/batch-0017.jsonl': '0aad71c296af1d68eb2c1ba7cb2a6ceb8c551e21dafaacf3834f386d1ffdde9b', 'generation_provenance/worker-2/batch-0018.jsonl': '1e96a2e86d3155e376ae080444cb1335f35844485c2e62806bb67e81b7941627', 'generation_provenance/worker-2/batch-0019.jsonl': 'f6682172aa58999712e613b411e852b38376665c762d1d2da63fd811fbad1c57', 'generation_provenance/worker-2/batch-0020.jsonl': '2e78be27e438f12585ec58de178a2730135c06abe9aaa2a6a1e5d379d63636ec', 'generation_provenance/worker-2/batch-0021.jsonl': 'bcb1154678f9695c0674a47ac413953c736ae754ec3643ff345b8628a48ab034', 'generation_provenance/worker-2/batch-0022.jsonl': '7af17e02aa8e309ef660bf02759c13a419eff05c0917ce380d470ded66e34370', 'generation_provenance/worker-2/batch-0023.jsonl': '5a9c81ec73ff974d3d544db3a649bb26cab34079a448c81735cb89554aa718f1', 'generation_provenance/worker-2/batch-0024.jsonl': '59282c798985e0865310e1ba86f3b1a766a6f235942857d5901a3e7ce68f7f58', 'generation_provenance/worker-2/batch-0025.jsonl': '4ac846e0e9412096f073cf5ade0d8988cb9627cc30c725fbb6a602340c535455', 'generation_provenance/worker-2/batch-0026.jsonl': '60a774a0167fd022fb7b364882f8e5c3f909377cf6ab98cb4852c09d06089590', 'generation_provenance/worker-2/batch-0027.jsonl': 'a6b3ace0f124c23acbd00ea73af13d96d531e9c406647567958cb6e98842caf1', 'generation_provenance/worker-2/batch-0028.jsonl': '5afd711d89471f333899801a798795dc8de7cc43db00269746bc2bf08eafd445', 'generation_provenance/worker-2/batch-0029.jsonl': '4adecba94af0d72e9cd62e7772d31c7ee4ad539942ed13d2746f6ee0cfe920b4', 'generation_provenance/worker-2/batch-0030.jsonl': 'ca63fc775853ebe054ba54ba763fbabaf680db22af6fff61b5745aaa97a92cf9', 'generation_provenance/worker-2/batch-0031.jsonl': 'fe70571f7925886aff0a91c10daaef8b8174669e05536fc2be08c4d7c2373796', 'generation_provenance/worker-2/batch-0032.jsonl': '2a81fb2f7ce119e9492a94f2c755f28275207b8fc7722da95bbcef3141a6421b', 'generation_provenance/worker-2/batch-0033.jsonl': 'd720e6c5598b1670b7cb839a25f8606545e2b5c6fc8161981b24da981963e6f3', 'generation_provenance/worker-2/batch-0034.jsonl': '0e06269dc33052efa63563c9821e723ab3489c21f783f37a40c48a81d54df138', 'generation_provenance/worker-2/batch-0035.jsonl': '92947f7177db007b7a9c10edf9680ae8bad93bb44c0109bf9dfec81baebe0000', 'generation_provenance/worker-2/batch-0036.jsonl': 'b534966c48ecca5b72ba9de3c44f08578515b84232a019b647e4797643cbffc0', 'generation_provenance/worker-2/batch-0037.jsonl': '5ddde9a4413c4dddf2890027ec203e715b5eecc18b0ef3e93bc5b92f0400f648', 'generation_provenance/worker-2/batch-0038.jsonl': 'ceb8970f800670ac8f1aaa2ce1c698a4af4e1a23b92c04384f86947fe8890bcf', 'generation_provenance/worker-2/batch-0039.jsonl': '30dcc1f37026aad72929be3f6b442ce12e0ec7d288fecbd0036fb654c754e2d6', 'generation_provenance/worker-2/batch-0040.jsonl': '542e70220824a9d6d493a1791a37f7308534336271a17459e1db6b33981dd4b1', 'generation_provenance/worker-2/batch-0041.jsonl': 'aeb1424e6a241f9eefe7c13387b0676e07a315939291a0b60feb8207a3ca82e6', 'generation_provenance/worker-2/batch-0042.jsonl': '76645f74af4244c8f47f0359337fd5828e7150f03e3944d69d79db894846a047', 'generation_provenance/worker-2/batch-0043.jsonl': 'bc90285c4d220fd4faac18e794343cb9fd44e03f39b4c72132b6eff1172e28a7', 'generation_provenance/worker-2/batch-0044.jsonl': '994720858254794741cefbf45553c9bc95ce4b746ad883d4652d62159390be39', 'generation_provenance/worker-2/batch-0045.jsonl': '4f1e0b3c91e78836b330492faea72ec13b8878db074dfff196880def0d2a8582', 'generation_provenance/worker-2/batch-0046.jsonl': '3fd5aeced79a3e91ee21bc5c7ec58c1932b2ffb99b7a726ea23a738a2ab3d1b7', 'generation_provenance/worker-2/batch-0047.jsonl': '90b77159df8ce1a6838f1f934823921613f9db67cff716fd46aad734e9542fc8', 'generation_provenance/worker-2/batch-0048.jsonl': '6917ed4a4221ebbf30091c2b279988003c3d3b8a2cf94447cb4a582b024c441d', 'generation_provenance/worker-2/batch-0049.jsonl': '72fdc6c02c2a8a58a30385933a2e38aea25d2840ddb29b52bd617ff067ed3fb0', 'generation_provenance/worker-2/batch-0050.jsonl': '1d3769b142682838e98f7e654a953d59b6e886b22fb45f100b815cbbeb108cc5', 'generation_provenance/worker-2/batch-0051.jsonl': 'e6842c83ad7a451517ffd019920a03ad1c670df7f0999ffe1283f3d6006fd462', 'generation_provenance/worker-2/batch-0052.jsonl': '7c4a89e5ccba3e8ba6c2d6388f41546273ff57f8fc868c72f875aadc2125975c', 'generation_provenance/worker-2/batch-0053.jsonl': '3d4f999ff7ae8ec62d61641392cb361b30b6c9589300827055bf6317cc36800e', 'generation_provenance/worker-2/batch-0054.jsonl': '6146187c6c395183d6c21db0ed2f5065997eb40fa714bccfdbcf7211c6e413a3', 'generation_provenance/worker-2/batch-0055.jsonl': '8c6bd81160b79393b7dcab1efa36bfd5e933221191185b38fc70b9e635ce0b68', 'generation_provenance/worker-2/batch-0056.jsonl': '19acc3482f3f2b9b86dbbd8d455921c880075332a8b69774e58e54a997c2133d', 'generation_provenance/worker-2/batch-0057.jsonl': '917fc7da224328efc6cea08b7bc19a7cff1cbdb802684882f9d350ba10a62657', 'generation_provenance/worker-2/batch-0058.jsonl': 'f35f3ac5c0bcbf52f094fd177cb50302ccf67771ecbe463c0d2ee679cbb67aa3', 'generation_provenance/worker-2/batch-0059.jsonl': 'a8727ed7c17dbf98f606d6efedb012a1230cc3e9bd2f5b52a186af710c4ecd69', 'generation_provenance/worker-2/batch-0060.jsonl': 'c9580f717138fa8a97595c3c9987f5839046bf2f8360c88dad9c105e80eb5c76', 'generation_provenance/worker-2/batch-0061.jsonl': '4655a0a1976447ca64dca91c4f8e200122940de2d28da0624bb36abd910b8a34', 'generation_provenance/worker-2/batch-0062.jsonl': '3bc1b2e77451943be0c68f477b0852a16d5a40a4d474780a99cb966f2aa95383', 'generation_provenance/worker-2/batch-0063.jsonl': 'e5c833b7b775ca426a9ee6a5fda63051d8680843e29feba91d7d27b4efa7c198', 'generation_provenance/worker-2/batch-0064.jsonl': '4f915af285989f9f59e78e4e92b61950a6351e897ab6fcb28d6925f0c860d3ec', 'generation_provenance/worker-2/batch-0065.jsonl': '30c10a359f50a5a723bc67dcb1934619e9f35c48a12a3535381bc22255bb2126', 'generation_provenance/worker-2/batch-0066.jsonl': '268ad722908ab99d7c620220b80d4593eeee3425e2a182af79ec560a62100250', 'generation_provenance/worker-2/batch-0067.jsonl': '9fc62b6380cc65ac0160efde7b6ab8bbb2a6ee96bc022cf7785c9851ac2972a1', 'generation_provenance/worker-2/batch-0068.jsonl': '7e835aec009789101cfc8233e40509e0f2458e5e801a9b62b89abf714eae9d48', 'generation_provenance/worker-2/batch-0069.jsonl': 'dbad740c887669e41d1e3909b2df489b6b3b30b8398066d9591abf5cca2db2a2', 'generation_provenance/worker-2/batch-0070.jsonl': '9edd0e147d0d914ccf3e3c01354f8317f8b6bc331142468037a05ad209a457aa', 'generation_provenance/worker-2/batch-0071.jsonl': 'a91d551e90ca8b54b56429f6ca7c299880ad02ec2df0f9296b031416b955278e', 'generation_provenance/worker-2/batch-0072.jsonl': 'f05939f816b138153088289453404d5eb8b6fc8e7d2a25d3f24def435c936f35', 'generation_provenance/worker-2/batch-0073.jsonl': '2255a4200703e9b719c21e33172443ede624add87c2ce42b1899614cc3a4132a', 'generation_provenance/worker-2/batch-0074.jsonl': 'a121128241704b71608880c1478ba33ca4e1e6cc8805cb5d5758d3f1dc5f36fd', 'generation_provenance/worker-2/evidence.jsonl': 'f01af2e085121703a41c74cd2fbf65b64021ec4c3beea2c2b3a6c1833b46736e', 'generation_provenance/worker-2/progress.json': '90601e939aec268665e1331a75ba0bbc5dbb527291c8548e3389f8004b92bfad', 'generation_provenance/worker-2/record-lineage.jsonl': '6cf11724d1b7124ced81c632b048d8bdec3a8d16e6f3d0666f62b13ec8a5d630', 'generation_provenance/worker-2/rejected-drafts/primary-before-primary-decisions-0005-batch-0000.jsonl': 'c6fecddaf0778609db14d17ffba30f7bbc69aa282e567eada852babbe0aff3eb', 'generation_provenance/worker-2/rejected-drafts/primary-before-primary-decisions-0005-batch-0001.jsonl': '391f4c51bbede3b1cb1528875e7889dbc2bd9f02952f6a45735946f9f06f7fa7', 'generation_provenance/worker-2/rejected-drafts/primary-before-primary-decisions-0006-batch-0001.jsonl': '26263719d26fb38ca7b7eb37618b1693991122ab4cca6b527072fd06c9a20591', 'generation_provenance/worker-2/source_audit.jsonl': 'b0a6d02a42ee614fef3d2ea1a1c9b57ff5b64342972f8f86500e4916af7e9f1f', 'generation_provenance/WORKER_INSTRUCTIONS.md': '5970b37fb1613c59650eb4e0b674f13672a98c39a9b257684edc2a1910b5bc29', 'starting_adapter/adapter_config.json': 'b52d635062584579f7fae9a5a4e2034c8dc4de93fa8e93024814cfe4e61df6bb', 'starting_adapter/adapter_model.safetensors': '11ccee4e737ddadc11258d5e3ac6820b82804036a7ca3b3f05ae45c4c44f9f95', 'starting_adapter/added_tokens.json': '58b54bbe36fc752f79a24a271ef66a0a0830054b4dfad94bde757d851968060b', 'starting_adapter/chat_template.jinja': 'cd8e9439f0570856fd70470bf8889ebd8b5d1107207f67a5efb46e342330527f', 'starting_adapter/merges.txt': '8831e4f1a044471340f7c0a83d7bd71306a5b867e95fd870f74d0c5308a904d5', 'starting_adapter/README.md': '2e326a70ff8cf6b8cc64186dbe97e6d7504d66291fd2e7de4c4578602e95657c', 'starting_adapter/special_tokens_map.json': '76862e765266b85aa9459767e33cbaf13970f327a0e88d1c65846c2ddd3a1ecd', 'starting_adapter/tokenizer.json': '9c5ae00e602b8860cbd784ba82a8aa14e8feecec692e7076590d014d7b7fdafa', 'starting_adapter/tokenizer_config.json': '60bf17d6543b028dbd97ac7f4847418c96502006b80b4b1c57427c241c818224', 'starting_adapter/vocab.json': 'ca10d7e9fb3ed18575dd1e277a2579c16d108e32f27439684afa0e10b1440910', 'TestData.csv': '5e10ec19d61624ee3554814337cf961fe499f4a666881c866c172fa1e51454a2', 'train.csv': '8d07ab7847bdc1e4f85f363122813a7d1a91d4ec850776bf552119bb80aacb20', 'validation.csv': 'c7c028971d4ba185113eefafbb74260db4e5484542c04fec96334e1b18c77e20'}
EXPECTED = {name: digest for name, digest in EXPECTED.items() if not name.startswith('starting_adapter/')}
matches = sorted({p.parent for p in Path('/kaggle/input').rglob('augmentation_manifest.json')
                  if all((p.parent / name).is_file() for name in EXPECTED)})
if len(matches) != 1:
    raise ValueError(f'Attach exactly one matching augmented dataset, found {matches}')
DATA = matches[0]
for name, digest in EXPECTED.items():
    if hashlib.sha256((DATA / name).read_bytes()).hexdigest() != digest:
        raise ValueError(f'Input hash mismatch: {name}')
    target = WORK / name
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(DATA / name, target)
MANIFEST = json.loads(Path('augmentation_manifest.json').read_text())
REVISION = MANIFEST['base_revision']
RESULT = WORK / ('qwen-fresh-base-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
RESULT.mkdir(exist_ok=False)


In [ ]:
p = Path('benchmark.py')
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""Zero-shot MCQ baseline. No training or training-data access."""\n\nimport argparse\nimport csv\nimport hashlib\nimport importlib.metadata\nimport json\nfrom pathlib import Path\nimport re\nimport time\n\n\ndef build_prompt(row):\n    choices = "\\n".join(f"{key}) {row[key]}" for key in "abcd")\n    return (\n        "Answer the following multiple-choice question about a Unity C# game project.\\n"\n        "Choose the single best answer. End your response with Answer:a, Answer:b, "\n        "Answer:c, or Answer:d.\\n\\n"\n        f"{row[\'Question\']}\\n{choices}"\n    )\n\n\ndef parse_answer(response):\n    # An unfinished thinking block is not a final answer.\n    if "<think>" in response and "</think>" not in response:\n        return None\n    final = response.rsplit("</think>", 1)[-1].strip()\n    match = re.search(r"Answer\\s*:\\s*([abcd])\\s*[.!]?\\s*$", final, re.I)\n    if match:\n        return match.group(1).lower()\n    return final.lower() if re.fullmatch(r"[abcd]", final, re.I) else None\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--model", default="Qwen/Qwen3-4B")\n    parser.add_argument("--data", type=Path, default=Path("TestData.csv"))\n    parser.add_argument("--output", type=Path, required=True,\n                        help="New directory for this run; existing directories are rejected.")\n    parser.add_argument("--limit", type=int, help="First N rows for smoke checks only")\n    parser.add_argument("--thinking", action="store_true", help="Qwen3 thinking mode")\n    parser.add_argument("--max-new-tokens", type=int)\n    parser.add_argument("--four-bit", action="store_true")\n    parser.add_argument("--seed", type=int, default=42)\n    args = parser.parse_args(argv)\n    if args.limit is not None and args.limit < 1:\n        parser.error("--limit must be positive")\n    if args.max_new_tokens is not None and args.max_new_tokens < 1:\n        parser.error("--max-new-tokens must be positive")\n    if args.thinking and not args.model.startswith("Qwen/Qwen3-"):\n        parser.error("--thinking is implemented only for Qwen3 models")\n    with args.data.open(encoding="utf-8-sig", newline="") as handle:\n        reader = csv.DictReader(handle)\n        required = {"class", "Question", "a", "b", "c", "d", "Answer"}\n        if not required.issubset(reader.fieldnames or []):\n            parser.error("Unexpected test CSV schema")\n        rows = list(reader)\n    if args.limit:\n        rows = rows[:args.limit]\n    if not rows or any(row["Answer"].strip().lower() not in "abcd" or\n                       len(row["Answer"].strip()) != 1 for row in rows):\n        parser.error("Empty dataset or invalid answer labels")\n\n    import torch\n    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed\n\n    if not torch.cuda.is_available():\n        raise RuntimeError("A CUDA GPU is required. Select a GPU runtime in Colab/Kaggle.")\n    args.output.mkdir(parents=True, exist_ok=False)\n    set_seed(args.seed)\n    tokenizer = AutoTokenizer.from_pretrained(args.model)\n    config = {"device_map": "auto", "torch_dtype": torch.float16}\n    if args.four_bit:\n        config["quantization_config"] = BitsAndBytesConfig(\n            load_in_4bit=True, bnb_4bit_quant_type="nf4",\n            bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,\n        )\n    model = AutoModelForCausalLM.from_pretrained(args.model, **config).eval()\n    max_tokens = args.max_new_tokens or (2048 if args.thinking else 128)\n    generation = {"max_new_tokens": max_tokens, "do_sample": args.thinking,\n                  "pad_token_id": tokenizer.pad_token_id or tokenizer.eos_token_id}\n    if args.thinking:\n        generation.update(temperature=0.6, top_p=0.95, top_k=20)\n    metadata = {\n        "model": args.model, "model_commit": getattr(model.config, "_commit_hash", None),\n        "dataset_sha256": hashlib.sha256(args.data.read_bytes()).hexdigest(),\n        "rows": len(rows), "smoke_test": args.limit is not None, "seed": args.seed,\n        "thinking": args.thinking, "four_bit": args.four_bit, "generation": generation,\n        "gpu": torch.cuda.get_device_name(0), "prompt_example": build_prompt(rows[0]),\n        "versions": {p: importlib.metadata.version(p) for p in\n                     ["torch", "transformers", "accelerate", "bitsandbytes"]},\n        "protocol": "Zero-shot; no training examples or source code; invalid answers count as wrong.",\n    }\n    (args.output / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")\n    predictions = []\n    start = time.perf_counter()\n    with (args.output / "predictions.jsonl").open("x", encoding="utf-8") as handle:\n        for index, row in enumerate(rows):\n            messages = [{"role": "user", "content": build_prompt(row)}]\n            template_args = {}\n            if args.model.startswith("Qwen/Qwen3-"):\n                template_args["enable_thinking"] = args.thinking\n            text = tokenizer.apply_chat_template(\n                messages, tokenize=False, add_generation_prompt=True, **template_args)\n            inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False).to(model.device)\n            tick = time.perf_counter()\n            with torch.inference_mode():\n                output = model.generate(**inputs, **generation)\n            generated = output[0, inputs.input_ids.shape[1]:]\n            response = tokenizer.decode(generated, skip_special_tokens=True)\n            answer = parse_answer(response)\n            result = {\n                "row": index, "class": row["class"], "question": row["Question"],\n                "response": response, "prediction": answer,\n                "truth": row["Answer"].strip().lower(),\n                "correct": answer == row["Answer"].strip().lower(),\n                "seconds": time.perf_counter() - tick,\n                "generated_tokens": len(generated),\n                "token_budget_reached": len(generated) >= max_tokens,\n            }\n            predictions.append(result)\n            handle.write(json.dumps(result, ensure_ascii=False) + "\\n")\n            handle.flush()\n            print(f"{index + 1}/{len(rows)} correct={sum(p[\'correct\'] for p in predictions)}", flush=True)\n    correct = sum(p["correct"] for p in predictions)\n    labels = {key: sum(p["truth"] == key for p in predictions) for key in "abcd"}\n    summary = {\n        "model": args.model, "questions": len(rows), "correct": correct,\n        "accuracy": correct / len(rows),\n        "invalid_answers": sum(p["prediction"] is None for p in predictions),\n        "token_budget_reached": sum(p["token_budget_reached"] for p in predictions),\n        "uniform_random_expected_accuracy": 0.25,\n        "label_counts": labels, "constant_choice_accuracy": {k: v / len(rows) for k, v in labels.items()},\n        "elapsed_seconds": time.perf_counter() - start,\n        "smoke_test": args.limit is not None,\n    }\n    (args.output / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")\n    print(json.dumps(summary, indent=2))\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')


In [ ]:
p = Path('finetune_qwen_coder.py')
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""One-epoch QLoRA experiment; test labels never enter training or prompts."""\n\nimport argparse\nfrom collections import defaultdict\nimport csv\nimport gc\nimport hashlib\nimport importlib.metadata\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport random\nimport re\nimport time\n\nos.environ.setdefault("USE_TF", "0")\nos.environ.setdefault("TOKENIZERS_PARALLELISM", "false")\n\nfrom benchmark import build_prompt, parse_answer\n\nMODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct"\n\n\ndef write_json(path, value):\n    Path(path).write_text(json.dumps(value, indent=2, ensure_ascii=False) + "\\n", encoding="utf-8")\n\n\ndef read_csv(path, required):\n    with Path(path).open(encoding="utf-8-sig", newline="") as handle:\n        reader = csv.DictReader(handle)\n        if not set(required).issubset(reader.fieldnames or []):\n            raise ValueError(f"Unexpected CSV schema: {path}")\n        rows = list(reader)\n    if not rows or any(not str(row.get(key) or "").strip() for row in rows for key in required):\n        raise ValueError(f"Empty rows or required fields: {path}")\n    return rows\n\n\ndef split_training_rows(rows, seed=42, validation_fraction=0.1):\n    # Keep identical questions together even if class names or answers differ.\n    groups = defaultdict(list)\n    for index, row in enumerate(rows):\n        question = re.sub(r"\\s+", " ", row["Q"].strip().casefold())\n        groups[question].append(index)\n    keys = sorted(groups)\n    random.Random(seed).shuffle(keys)\n    validation_keys = set(keys[:max(1, round(len(keys) * validation_fraction))])\n    validation_indices = sorted(i for key in validation_keys for i in groups[key])\n    validation_set = set(validation_indices)\n    training_indices = [i for i in range(len(rows)) if i not in validation_set]\n    if not training_indices or not validation_indices:\n        raise ValueError("Need enough distinct questions for train/validation splitting")\n    return training_indices, validation_indices\n\n\ndef tokenize_example(row, tokenizer, max_length):\n    user = f"In this Unity game project, regarding {row[\'class\'].strip()}:\\n{row[\'Q\'].strip()}"\n    messages = [{"role": "user", "content": user}]\n    prefix = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True)\n    full = tokenizer.apply_chat_template(\n        messages + [{"role": "assistant", "content": row["A"].strip()}],\n        tokenize=True, add_generation_prompt=False,\n    )\n    if full[:len(prefix)] != prefix:\n        raise ValueError("Chat-template prefix mismatch; refusing incorrect loss masking")\n    if len(full) > max_length:\n        return None, len(full)\n    labels = [-100] * len(prefix) + full[len(prefix):]\n    if not any(label != -100 for label in labels):\n        raise ValueError("Example contains no answer tokens")\n    return {"input_ids": full, "attention_mask": [1] * len(full), "labels": labels}, len(full)\n\n\ndef collate_examples(examples, pad_token_id):\n    import torch\n    width = math.ceil(max(len(row["input_ids"]) for row in examples) / 8) * 8\n    batch = {key: [] for key in ["input_ids", "attention_mask", "labels"]}\n    for row in examples:\n        padding = width - len(row["input_ids"])\n        for key, pad in [("input_ids", pad_token_id), ("attention_mask", 0), ("labels", -100)]:\n            batch[key].append(row[key] + [pad] * padding)\n    return {key: torch.tensor(value, dtype=torch.long) for key, value in batch.items()}\n\n\ndef evaluate_mcq(model, tokenizer, rows, output):\n    import torch\n    output.mkdir(parents=True, exist_ok=False)\n    model.eval()\n    model.config.use_cache = True\n    predictions = []\n    torch.cuda.synchronize()\n    start = time.perf_counter()\n    with (output / "predictions.jsonl").open("x", encoding="utf-8") as handle:\n        for index, row in enumerate(rows):\n            prompt = build_prompt(row)\n            text = tokenizer.apply_chat_template(\n                [{"role": "user", "content": prompt}],\n                tokenize=False, add_generation_prompt=True,\n            )\n            inputs = tokenizer(text, add_special_tokens=False, return_tensors="pt").to(model.device)\n            tick = time.perf_counter()\n            with torch.inference_mode():\n                generated = model.generate(\n                    **inputs, do_sample=False, max_new_tokens=128,\n                    pad_token_id=tokenizer.pad_token_id,\n                )[0, inputs.input_ids.shape[1]:]\n            response = tokenizer.decode(generated, skip_special_tokens=True)\n            answer = parse_answer(response)\n            result = {\n                "row": index, "class": row["class"], "question": row["Question"],\n                "response": response, "prediction": answer, "truth": row["Answer"].strip().lower(),\n                "correct": answer == row["Answer"].strip().lower(),\n                "generated_tokens": len(generated), "token_budget_reached": len(generated) >= 128,\n                "seconds": time.perf_counter() - tick,\n            }\n            predictions.append(result)\n            handle.write(json.dumps(result, ensure_ascii=False) + "\\n")\n            handle.flush()\n            if (index + 1) % 10 == 0 or index + 1 == len(rows):\n                print(f"{output.name}: {index + 1}/{len(rows)}", flush=True)\n    summary = {\n        "model": MODEL_ID, "questions": len(rows),\n        "correct": sum(row["correct"] for row in predictions),\n        "accuracy": sum(row["correct"] for row in predictions) / len(rows),\n        "invalid_answers": sum(row["prediction"] is None for row in predictions),\n        "token_budget_reached": sum(row["token_budget_reached"] for row in predictions),\n        "elapsed_seconds": time.perf_counter() - start,\n    }\n    write_json(output / "summary.json", summary)\n    with (output / "predictions.csv").open("x", encoding="utf-8", newline="") as handle:\n        writer = csv.DictWriter(handle, fieldnames=list(predictions[0]))\n        writer.writeheader()\n        writer.writerows(predictions)\n    return summary, predictions\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--train", type=Path, default=Path("TrainData.csv"))\n    parser.add_argument("--test", type=Path, default=Path("TestData.csv"))\n    parser.add_argument("--output", type=Path, required=True)\n    parser.add_argument("--max-length", type=int, default=2048)\n    parser.add_argument("--seed", type=int, default=42)\n    args = parser.parse_args()\n    if args.max_length < 64:\n        parser.error("--max-length must be at least 64")\n    train_rows = read_csv(args.train, ["class", "Q", "A"])\n    test_rows = read_csv(args.test, ["class", "Question", "a", "b", "c", "d", "Answer"])\n    if any(row["Answer"].strip().lower() not in list("abcd") for row in test_rows):\n        raise ValueError("Invalid MCQ answer label")\n\n    import torch\n    from functools import partial\n    from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training\n    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, Trainer, TrainingArguments, set_seed\n\n    if not torch.cuda.is_available():\n        raise RuntimeError("Enable a CUDA GPU before running this experiment")\n    args.output.mkdir(parents=True, exist_ok=False)\n    set_seed(args.seed)\n    training_indices, validation_indices = split_training_rows(train_rows, args.seed)\n    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)\n    if tokenizer.pad_token_id is None:\n        tokenizer.pad_token = tokenizer.eos_token\n    tokenizer.padding_side = "right"\n    prepared = {}\n    lengths = []\n    exclusions = []\n    for index, row in enumerate(train_rows):\n        example, length = tokenize_example(row, tokenizer, args.max_length)\n        lengths.append(length)\n        if example is None:\n            exclusions.append({"row": index, "class": row["class"], "tokens": length,\n                               "reason": "Exceeds max_length; excluded without truncating answers"})\n        else:\n            prepared[index] = example\n    training_indices = [i for i in training_indices if i in prepared]\n    validation_indices = [i for i in validation_indices if i in prepared]\n    if not training_indices or not validation_indices:\n        raise ValueError("No usable training or validation examples after length checks")\n    write_json(args.output / "split.json", {\n        "seed": args.seed, "train_indices": training_indices,\n        "validation_indices": validation_indices, "excluded": exclusions,\n        "note": "Grouped by identical normalized question. Semantic paraphrases may still overlap.",\n    })\n    quantization = BitsAndBytesConfig(\n        load_in_4bit=True, bnb_4bit_quant_type="nf4",\n        bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,\n    )\n    model = AutoModelForCausalLM.from_pretrained(\n        MODEL_ID, quantization_config=quantization, dtype=torch.float16,\n        device_map={"": 0}, attn_implementation="sdpa",\n    )\n    revision = getattr(model.config, "_commit_hash", None)\n    training_settings = {\n        "num_train_epochs": 1, "learning_rate": 1e-4, "batch_size": 1,\n        "gradient_accumulation_steps": 16, "warmup_ratio": 0.05,\n        "lr_scheduler_type": "cosine", "weight_decay": 0.01,\n        "max_grad_norm": 0.3, "lora_r": 16, "lora_alpha": 32,\n        "lora_dropout": 0.05, "max_length": args.max_length,\n    }\n    metadata = {\n        "model": MODEL_ID, "model_commit": revision, "seed": args.seed,\n        "gpu": torch.cuda.get_device_name(0), "gpu_count_used": 1,\n        "train_sha256": hashlib.sha256(args.train.read_bytes()).hexdigest(),\n        "test_sha256": hashlib.sha256(args.test.read_bytes()).hexdigest(),\n        "train_rows": len(train_rows), "train_examples_used": len(training_indices),\n        "validation_examples": len(validation_indices), "test_rows": len(test_rows),\n        "excluded_long_examples": len(exclusions),\n        "train_token_lengths": {"min": min(lengths), "max": max(lengths), "mean": sum(lengths)/len(lengths)},\n        "settings": training_settings,\n        "versions": {p: importlib.metadata.version(p) for p in\n                     ["torch", "transformers", "peft", "accelerate", "bitsandbytes"]},\n        "protocol": "One epoch on training split only. Answer-only loss. Test evaluated once before and once after training, never used for selection. Fresh base and adapter share decoding and packages.",\n        "evaluation": {"max_new_tokens": 128, "do_sample": False,\n                       "invalid_answers_count_as_wrong": True, "prompt_example": build_prompt(test_rows[0])},\n    }\n    write_json(args.output / "metadata.json", metadata)\n    print(f"Training: {len(training_indices)}; validation: {len(validation_indices)}; exclusions: {len(exclusions)}", flush=True)\n    print("Measuring a fresh base-model reference with the same environment...", flush=True)\n    base_summary, base_predictions = evaluate_mcq(model, tokenizer, test_rows, args.output / "base")\n\n    # Reset stochastic state so training does not depend on evaluation RNG consumption.\n    set_seed(args.seed)\n    model.config.use_cache = False\n    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)\n    model = get_peft_model(model, LoraConfig(\n        r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",\n        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],\n    ))\n    model.print_trainable_parameters()\n    trainable, total = model.get_nb_trainable_parameters()\n    metadata.update(trainable_parameters=trainable, total_parameters=total)\n    write_json(args.output / "metadata.json", metadata)\n    training_args = TrainingArguments(\n        output_dir=str(args.output / "training"), num_train_epochs=1,\n        learning_rate=1e-4, per_device_train_batch_size=1, per_device_eval_batch_size=1,\n        gradient_accumulation_steps=16, gradient_checkpointing=True,\n        gradient_checkpointing_kwargs={"use_reentrant": False},\n        fp16=True, optim="paged_adamw_8bit", lr_scheduler_type="cosine",\n        warmup_ratio=0.05, weight_decay=0.01, max_grad_norm=0.3,\n        logging_steps=5, eval_strategy="epoch", save_strategy="no",\n        prediction_loss_only=True, report_to="none", seed=args.seed, data_seed=args.seed,\n        dataloader_num_workers=0, remove_unused_columns=False,\n    )\n    trainer = Trainer(\n        model=model, args=training_args,\n        train_dataset=[prepared[i] for i in training_indices],\n        eval_dataset=[prepared[i] for i in validation_indices],\n        processing_class=tokenizer,\n        data_collator=partial(collate_examples, pad_token_id=tokenizer.pad_token_id),\n    )\n    torch.cuda.reset_peak_memory_stats()\n    trained = trainer.train()\n    write_json(args.output / "training_metrics.json", {\n        **trained.metrics, "peak_gpu_allocated_gib": torch.cuda.max_memory_allocated()/1024**3,\n    })\n    write_json(args.output / "training_log.json", trainer.state.log_history)\n    adapter_path = args.output / "adapter"\n    model.save_pretrained(adapter_path, safe_serialization=True)\n    tokenizer.save_pretrained(adapter_path)\n\n    # Reload from disk before final scoring, verifying adapter portability.\n    del trainer, model\n    gc.collect()\n    torch.cuda.empty_cache()\n    base = AutoModelForCausalLM.from_pretrained(\n        MODEL_ID, revision=revision, quantization_config=quantization,\n        dtype=torch.float16, device_map={"": 0}, attn_implementation="sdpa",\n    )\n    model = PeftModel.from_pretrained(base, str(adapter_path), is_trainable=False)\n    tuned_summary, tuned_predictions = evaluate_mcq(model, tokenizer, test_rows, args.output / "finetuned")\n    comparison = {\n        "base": base_summary, "finetuned": tuned_summary,\n        "accuracy_change_percentage_points": 100*(tuned_summary["accuracy"]-base_summary["accuracy"]),\n        "relative_accuracy_change": (tuned_summary["accuracy"]-base_summary["accuracy"])/base_summary["accuracy"] if base_summary["accuracy"] else None,\n        "wrong_to_correct": sum(not a["correct"] and b["correct"] for a,b in zip(base_predictions,tuned_predictions)),\n        "correct_to_wrong": sum(a["correct"] and not b["correct"] for a,b in zip(base_predictions,tuned_predictions)),\n        "saved_adapter_reload_verified": True,\n    }\n    write_json(args.output / "comparison.json", comparison)\n    print(json.dumps(comparison, indent=2), flush=True)\n    print(f"Completed. Adapter and results: {args.output}", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')


In [ ]:
p = Path('finetune_qwen_mcq.py')
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""MCQ QLoRA continuation with configurable option presentations."""\n\nimport argparse\nfrom collections import Counter\nfrom functools import partial\nimport gc\nimport hashlib\nimport importlib.metadata\nimport json\nimport math\nimport os\nimport platform\nfrom pathlib import Path\nimport random\nimport re\n\n# Set before torch initializes in the training subprocess.\nos.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")\n\nfrom benchmark import build_prompt\nfrom finetune_qwen_coder import MODEL_ID, collate_examples, evaluate_mcq, read_csv, write_json\n\nLETTERS = "abcd"\nMCQ_FIELDS = ["class", "Question", *LETTERS, "Answer"]\n\n\ndef assistant_only_loss(model, inputs, return_outputs=False, num_items_in_batch=None):\n    """Exact masked causal loss without materializing prompt vocabulary logits.\n\n    Qwen\'s logits_to_keep slices hidden states before the vocabulary projection.\n    Keep the predecessor of each supervised label, including the turn ending.\n    This also works with padding and different answer positions within a batch.\n    """\n    import torch\n    labels = inputs["labels"]\n    targets = labels[:, 1:]\n    positions = torch.where((targets != -100).any(dim=0))[0]\n    if positions.numel() == 0:\n        raise ValueError("Batch has no supervised next-token targets")\n    forward_inputs = {k: v for k, v in inputs.items() if k != "labels"}\n    outputs = model(**forward_inputs, logits_to_keep=positions, use_cache=False)\n    selected_targets = targets.index_select(1, positions).to(outputs.logits.device)\n    loss = torch.nn.functional.cross_entropy(\n        outputs.logits.float().reshape(-1, outputs.logits.shape[-1]),\n        selected_targets.reshape(-1), ignore_index=-100,\n        reduction="sum" if num_items_in_batch is not None else "mean",\n    )\n    if num_items_in_batch is not None:\n        # Preserve Trainer\'s token normalization across gradient accumulation.\n        if isinstance(num_items_in_batch, torch.Tensor):\n            num_items_in_batch = num_items_in_batch.to(loss.device)\n        loss = loss / num_items_in_batch\n    return (loss, outputs) if return_outputs else loss\n\n\ndef load_trainable_adapter(model, adapter):\n    """Load original learned LoRA tensors and verify exact warm-start values."""\n    import torch\n    from peft import PeftModel, get_peft_model_state_dict\n    from safetensors.torch import load_file\n    model = PeftModel.from_pretrained(model, str(adapter), is_trainable=True)\n    expected = load_file(str(Path(adapter) / "adapter_model.safetensors"))\n    actual = get_peft_model_state_dict(model)\n    if expected.keys() != actual.keys():\n        raise ValueError("Warm-start adapter tensor names differ")\n    for name, tensor in actual.items():\n        if not torch.equal(tensor.detach().cpu(), expected[name].to(tensor.dtype)):\n            raise ValueError(f"Warm-start adapter tensor differs: {name}")\n    if any(p.requires_grad for n, p in model.named_parameters() if "lora_" not in n):\n        raise ValueError("Base weights must remain frozen")\n    if not any(p.requires_grad for n, p in model.named_parameters() if "lora_" in n):\n        raise ValueError("Loaded LoRA adapter is not trainable")\n    return model\n\n\ndef load_split(path, split, allow_variants=False, validate_unique_options=True):\n    rows = read_csv(path, MCQ_FIELDS + ["source_row", "split"])\n    ids = set()\n    for row in rows:\n        if row["split"] != split:\n            raise ValueError(f"Wrong split in {path}")\n        try:\n            source_id = int(row["source_row"])\n        except ValueError as exc:\n            raise ValueError("Invalid source_row") from exc\n        variant_id = int(row.get("variant_id", "0"))\n        identity = (source_id, variant_id) if allow_variants else source_id\n        if variant_id not in (0, 1, 2) or (variant_id and not allow_variants):\n            raise ValueError("Invalid variant_id for this split")\n        if source_id < 0 or identity in ids:\n            raise ValueError("Invalid or repeated source_row within a split")\n        ids.add(identity)\n        row["Answer"] = row["Answer"].strip().lower()\n        if row["Answer"] not in list(LETTERS):\n            raise ValueError("Invalid answer letter")\n        if validate_unique_options and len({row[letter].strip() for letter in LETTERS}) != 4:\n            raise ValueError("Duplicate MCQ options")\n    return rows\n\n\ndef check_split_isolation(train, validation):\n    if {int(r["source_row"]) for r in train} & {int(r["source_row"]) for r in validation}:\n        raise ValueError("Source ids overlap between train and validation")\n    key = lambda r: re.sub(r"\\s+", " ", r["Question"].strip()).casefold()\n    if {key(r) for r in train} & {key(r) for r in validation}:\n        raise ValueError("Identical normalized questions cross splits")\n\n\ndef rotate_question(row, seed=42):\n    """Shuffle once per stable source id, then rotate all four options together."""\n    order = list(LETTERS)\n    seed_bytes = f"mcq-rotation-v1:{seed}:{row[\'source_row\']}".encode()\n    variant_id = int(row.get("variant_id", 0))\n    if variant_id:\n        seed_bytes = f"mcq-rotation-variant-v1:{seed}:{row[\'source_row\']}:{variant_id}".encode()\n    random.Random(int.from_bytes(hashlib.sha256(seed_bytes).digest(), "big")).shuffle(order)\n    result = []\n    for rotation in range(4):\n        permutation = order[rotation:] + order[:rotation]\n        mapping = dict(zip(LETTERS, permutation))  # new letter -> original letter\n        answer = next(new for new, old in mapping.items() if old == row["Answer"])\n        rotated = {**row, **{new: row[old] for new, old in mapping.items()}, "Answer": answer}\n        result.append((rotated, {"source_row": int(row["source_row"]), "split": row["split"],\n                                  **({"variant_id": variant_id} if "variant_id" in row else {}),\n                                  "rotation": rotation, "new_to_original": mapping,\n                                  "original_answer": row["Answer"], "answer": answer}))\n    return result\n\n\ndef tokenize_mcq(row, tokenizer, max_length):\n    messages = [{"role": "user", "content": build_prompt(row)}]\n    prefix = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_dict=False)\n    full = tokenizer.apply_chat_template(\n        messages + [{"role": "assistant", "content": "Answer:" + row["Answer"]}],\n        tokenize=True, add_generation_prompt=False, return_dict=False,\n    )\n    if full[:len(prefix)] != prefix:\n        raise ValueError("Chat-template prefix mismatch; refusing incorrect loss masking")\n    if len(full) <= len(prefix):\n        raise ValueError("No assistant target tokens")\n    if len(full) > max_length:\n        return None, len(full)\n    return {"input_ids": full, "attention_mask": [1] * len(full),\n            "labels": [-100] * len(prefix) + full[len(prefix):]}, len(full)\n\n\ndef prepare_split(rows, tokenizer, max_length, seed, training, presentation_mode="four_rotations"):\n    if presentation_mode not in ("four_rotations", "single_random"):\n        raise ValueError("Unknown training presentation mode")\n    prepared, admitted, maps, exclusions = [], [], [], []\n    for row in rows:\n        variants = rotate_question(row, seed)\n        if training and presentation_mode == "single_random":\n            variants = variants[:1]\n        encoded = [tokenize_mcq(v, tokenizer, max_length) for v, _ in variants]\n        fixed, fixed_length = encoded[0] if training and presentation_mode == "single_random" else tokenize_mcq(row, tokenizer, max_length)\n        lengths = [length for _, length in encoded]\n        # Use the same length-eligible validation cohort for fixed and rotated metrics.\n        if any(example is None for example, _ in encoded) or fixed is None:\n            exclusions.append({"source_row": int(row["source_row"]), "split": row["split"],\n                               "rotation_lengths": lengths, "fixed_length": fixed_length,\n                               "reason": "Entire question group exceeds max_length; no truncation"})\n            continue\n        admitted.append(row)\n        selected = range(len(variants))\n        if training:\n            prepared.extend(encoded[i][0] for i in selected)\n        else:\n            prepared.append(fixed)\n        for i in selected:\n            maps.append({**variants[i][1], "tokens": lengths[i]})\n    if not admitted:\n        raise ValueError("No questions fit max_length in a split")\n    if training:\n        random.Random(seed).shuffle(prepared)\n        if presentation_mode == "four_rotations":\n            assert Counter(m["answer"] for m in maps) == {l: len(admitted) for l in LETTERS}\n        else:\n            assert len(prepared) == len(maps) == len(admitted)\n    return prepared, admitted, maps, exclusions\n\n\ndef smoke_training_rows(rows, tokenizer, max_length, seed, presentation_mode="four_rotations"):\n    """Include the longest fitting question so smoke tests exercise peak lengths."""\n    _, admitted, maps, _ = prepare_split(rows, tokenizer, max_length, seed, True, presentation_mode)\n    longest_map = max(maps, key=lambda m: m["tokens"])\n    longest = next(r for r in admitted if int(r["source_row"]) == longest_map["source_row"]\n                   and int(r.get("variant_id", 0)) == longest_map.get("variant_id", 0))\n    return [longest] + [r for r in admitted if r is not longest][:7]\n\n\ndef rotation_diagnostics(rows, predictions, seed):\n    if len(predictions) != 4 * len(rows):\n        raise ValueError("Incomplete rotated predictions")\n    details, by_position = [], {l: {"correct": 0, "questions": 0} for l in LETTERS}\n    for index, row in enumerate(rows):\n        selected = []\n        correct = []\n        for offset, (_, mapping) in enumerate(rotate_question(row, seed)):\n            prediction = predictions[4 * index + offset]\n            letter = prediction["prediction"]\n            selected.append(mapping["new_to_original"].get(letter))\n            correct.append(prediction["correct"])\n            stat = by_position[mapping["answer"]]\n            stat["questions"] += 1\n            stat["correct"] += prediction["correct"]\n        details.append({"source_row": int(row["source_row"]),\n                        "selected_original_letters": selected,\n                        "consistent": None not in selected and len(set(selected)) == 1,\n                        "all_four_correct": all(correct), "correct_rotations": sum(correct)})\n    for stat in by_position.values():\n        stat["accuracy"] = stat["correct"] / stat["questions"]\n    return {"questions": len(rows), "accuracy_by_correct_position": by_position,\n            "consistent_answer_text_rate": sum(r["consistent"] for r in details) / len(rows),\n            "all_four_correct_rate": sum(r["all_four_correct"] for r in details) / len(rows),\n            "note": "Invalid responses are inconsistent. Consistency alone does not imply correctness.",\n            "per_question": details}\n\n\ndef score(model, tokenizer, validation, test, output, seed):\n    # Do not build a graph or retain KV caching during loss evaluation in Trainer.\n    fixed, _ = evaluate_mcq(model, tokenizer, validation, output / "validation")\n    test_summary, test_predictions = evaluate_mcq(model, tokenizer, test, output / "test")\n    rotated = [v for row in validation for v, _ in rotate_question(row, seed)]\n    rotated_summary, predictions = evaluate_mcq(model, tokenizer, rotated, output / "validation_rotated")\n    diagnostic = rotation_diagnostics(validation, predictions, seed)\n    diagnostic["aggregate"] = rotated_summary\n    write_json(output / "rotation_diagnostics.json", diagnostic)\n    return {"validation": fixed, "test": test_summary, "rotation": diagnostic}, test_predictions\n\n\ndef make_comparison(base_summary, base_predictions, tuned_summary, tuned_predictions, smoke_test=False):\n    result = {"base": base_summary, "finetuned": tuned_summary, "smoke_test": smoke_test,\n              "baseline_evaluation": "skipped" if base_summary is None else "measured_this_run",\n              "test_accuracy_change_percentage_points": None,\n              "wrong_to_correct": None, "correct_to_wrong": None,\n              "saved_adapter_reload_verified": True}\n    if base_summary is not None:\n        result.update(\n            test_accuracy_change_percentage_points=100 * (tuned_summary["test"]["accuracy"] - base_summary["test"]["accuracy"]),\n            wrong_to_correct=sum(not a["correct"] and b["correct"] for a, b in zip(base_predictions, tuned_predictions)),\n            correct_to_wrong=sum(a["correct"] and not b["correct"] for a, b in zip(base_predictions, tuned_predictions)),\n        )\n    return result\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--train", type=Path, default=Path("data/mcq/generated-mcqs-final/train.csv"))\n    parser.add_argument("--validation", type=Path, default=Path("data/mcq/generated-mcqs-final/validation.csv"))\n    parser.add_argument("--test", type=Path, default=Path("TestData.csv"))\n    parser.add_argument("--output", type=Path, required=True)\n    parser.add_argument("--max-length", type=int, default=4096)\n    parser.add_argument("--seed", type=int, default=42)\n    parser.add_argument("--revision", help="Optional base/tokenizer commit pin for a matched experiment")\n    parser.add_argument("--starting-adapter", type=Path, help="Trainable warm start; fresh optimizer/scheduler")\n    parser.add_argument("--fresh-base", action="store_true", help="Initialize fresh LoRA on original pretrained weights; never load a prior adapter")\n    parser.add_argument("--augmentation-manifest", type=Path, help="Require complete accepted augmentation and exact inputs")\n    parser.add_argument("--presentation-mode", choices=["four_rotations", "single_random"],\n                        help="Defaults to the augmentation manifest, otherwise legacy four rotations")\n    parser.add_argument("--checkpoint-evaluations", type=int, choices=[0, 3], default=None,\n                        help="Save weights and validation loss/accuracy at each third of the epoch")\n    parser.add_argument("--prepare-only", action="store_true", help="Tokenizer preflight; no model load or GPU required")\n    parser.add_argument("--preflight-tokenizer", type=Path, help="Local saved tokenizer for offline --prepare-only checks")\n    parser.add_argument("--smoke-test", action="store_true", help="8 train questions, 4 validation, 2 test; not benchmark results")\n    parser.add_argument("--skip-base-evaluation", action="store_true", help="Train immediately; score only the saved/reloaded fine-tuned adapter")\n    parser.add_argument("--alignment-qa", type=Path, help="Enable simplified alignment using verbatim original class,Q,A CSV; omit for unchanged control")\n    parser.add_argument("--max-reference-length", type=int, default=16384, help="Fail above this reference length; never truncate QA or exclude extra groups")\n    args = parser.parse_args(argv)\n    if args.fresh_base and (args.starting_adapter or args.alignment_qa):\n        parser.error("--fresh-base cannot use a starting adapter or alignment")\n    if args.max_length < 64:\n        parser.error("max-length must be at least 64")\n    if args.preflight_tokenizer and not args.prepare_only:\n        parser.error("--preflight-tokenizer is only allowed with --prepare-only")\n    if args.starting_adapter and (not args.augmentation_manifest or args.alignment_qa or not args.skip_base_evaluation):\n        parser.error("Warm start requires accepted augmentation, no alignment, and --skip-base-evaluation")\n    augmentation = None\n    if args.augmentation_manifest:\n        from mcq_augmentation import verify_training_manifest\n        augmentation = verify_training_manifest(args.augmentation_manifest, args.train, args.validation,\n                                                args.test, args.starting_adapter, args.revision, fresh_base=args.fresh_base)\n        if args.max_length != 4096 or args.seed != 42:\n            parser.error("Augmentation requires the 4096-token, seed-42 protocol")\n    manifest_mode = augmentation.get("training_presentation_mode", "four_rotations") if augmentation else "four_rotations"\n    mode = args.presentation_mode or manifest_mode\n    if augmentation and mode != manifest_mode:\n        parser.error("Presentation mode differs from dataset manifest")\n    checkpoint_evaluations = args.checkpoint_evaluations\n    if checkpoint_evaluations is None:\n        checkpoint_evaluations = augmentation.get("checkpoint_evaluations", 0) if augmentation else 0\n    if args.alignment_qa and (mode != "four_rotations" or checkpoint_evaluations):\n        parser.error("Alignment uses its existing four-rotation, epoch-evaluation protocol")\n    train = load_split(args.train, "train", allow_variants=augmentation is not None,\n                       validate_unique_options=not (augmentation and augmentation.get("acceptance_policy", {}).get("mode") == "as_generated"))\n    validation = load_split(args.validation, "validation")\n    check_split_isolation(train, validation)\n    total_train, total_validation = len(train), len(validation)\n    total_train_sources = len({int(r["source_row"]) for r in train})\n    args.output.mkdir(parents=True, exist_ok=False)\n    from transformers import AutoTokenizer, set_seed\n    set_seed(args.seed)\n    tokenizer = AutoTokenizer.from_pretrained(str(args.preflight_tokenizer) if args.preflight_tokenizer else MODEL_ID,\n                                              revision=args.revision, local_files_only=bool(args.preflight_tokenizer))\n    if tokenizer.pad_token_id is None:\n        tokenizer.pad_token = tokenizer.eos_token\n    tokenizer.padding_side = "right"\n    original_qa = None\n    if args.alignment_qa:\n        from mcq_alignment import load_original_qa\n        original_qa = load_original_qa(args.alignment_qa)\n    if args.smoke_test:\n        if original_qa is None:\n            train = smoke_training_rows(train, tokenizer, args.max_length, args.seed, mode)\n        else:\n            from mcq_alignment import alignment_smoke_rows\n            train = alignment_smoke_rows(train, original_qa, tokenizer, args.max_length, args.max_reference_length, args.seed)\n        validation = validation[:4]\n    train_data, train, train_maps, train_exclusions = prepare_split(train, tokenizer, args.max_length, args.seed, True, mode)\n    val_data, validation, val_maps, val_exclusions = prepare_split(validation, tokenizer, args.max_length, args.seed, False)\n    if augmentation is not None and (train_exclusions or val_exclusions):\n        raise ValueError("Reviewed augmentation/evaluation cohort must not lose any examples")\n    alignment_lengths = None\n    if original_qa is not None:\n        from mcq_alignment import attach_references\n        alignment_lengths = attach_references(train_data, train, original_qa, tokenizer, args.max_reference_length, args.seed)\n        write_json(args.output / "alignment_lengths.json", alignment_lengths)\n    metadata = {\n        "model": MODEL_ID, "seed": args.seed, "smoke_test": args.smoke_test,\n        "tokenizer_source": str(args.preflight_tokenizer) if args.preflight_tokenizer else MODEL_ID,\n        "source_counts": {"train": total_train_sources, "validation": total_validation},\n        "input_mcq_counts": {"train": total_train, "validation": total_validation},\n        "admitted_sources": {"train": len({int(r["source_row"]) for r in train}), "validation": len(validation)},\n        "admitted_questions": {"train": len(train), "validation": len(validation)},\n        "training_presentations": len(train_data), "train_label_counts": dict(Counter(m["answer"] for m in train_maps)),\n        "training_source_rows": sorted({int(r["source_row"]) for r in train}),\n        "input_sha256": {name: hashlib.sha256(path.read_bytes()).hexdigest() for name, path in\n                         [("train", args.train), ("validation", args.validation)]},\n        "baseline_evaluation": "skipped" if args.skip_base_evaluation else "measure_this_run",\n        "protocol": "One epoch of four rotations per question (four original-question exposures). Fixed validation plus rotation diagnostics after adapter reload. Baseline scoring is optional; skipped baselines are never fabricated or imported from unverified prior runs. Test labels never enter training. Not the original single-exposure assessment budget.",\n        "training_presentation_mode": mode, "checkpoint_evaluations": checkpoint_evaluations,\n        "settings": {"rotations": 1 if mode == "single_random" else 4, "max_length": args.max_length, "num_train_epochs": 1,\n                     "learning_rate": 1e-4, "batch_size": 1, "gradient_accumulation_steps": 16,\n                     "lora_r": 16, "lora_alpha": 32, "lora_dropout": 0.05,\n                     "target_format": "Answer:<letter>", "loss": "assistant response only",\n                     "loss_implementation": "selected causal logits before vocabulary projection",\n                     "allocator": os.environ.get("PYTORCH_CUDA_ALLOC_CONF")},\n        "evaluation": {"max_new_tokens": 128, "do_sample": False, "invalid_answers_count_as_wrong": True,\n                       "prompt_example": build_prompt(validation[0]),\n                       "validation_cohort": "All four orders and original order must fit max_length; full exclusions in length_exclusions.json"},\n    }\n    metadata["budget"] = {"total_input_tokens": sum(len(x["input_ids"]) for x in train_data),\n                          "supervised_tokens": sum(sum(t != -100 for t in x["labels"]) for x in train_data),\n                          "optimizer_steps": math.ceil(len(train_data) / 16),\n                          "longest_presentation_tokens": max(len(x["input_ids"]) for x in train_data)}\n    if augmentation is not None:\n        metadata["augmentation"] = augmentation\n        metadata["starting_adapter_sha256"] = None if args.fresh_base else augmentation["starting_adapter_sha256"]\n        metadata["optimizer_state"] = "fresh; no checkpoint resume"\n        metadata["protocol"] = ("One additional epoch from original MCQ adapter; originals plus two variants; "\n                                + ("one independently shuffled option order per MCQ" if mode == "single_random" else "four rotations per MCQ")\n                                + ". Training examples shuffled. Alignment disabled. Additional training and augmentation are confounded; no paraphrasing-only causal claim.")\n    metadata["initialization"] = "fresh_base_fresh_lora" if args.fresh_base else ("warm_start_adapter" if args.starting_adapter else "fresh_base_fresh_lora")\n    if args.fresh_base:\n        metadata["protocol"] = "One epoch from original pretrained Qwen weights with fresh LoRA and optimizer; no previous adapter. Originals plus two variants, one shuffled order per MCQ. Three within-epoch validation measurements."\n    if checkpoint_evaluations:\n        metadata["checkpoint_protocol"] = "Save adapter weights and evaluate identical fixed validation loss/accuracy near 1/3, 2/3 and the end of one epoch. Test only after final checkpoint; no automatic checkpoint selection."\n    write_json(args.output / "metadata.json", metadata)\n    if original_qa is not None:\n        metadata["alignment"] = {\n            "weight": 0.05, "description": "Simplified QA-conditioned cosine alignment; not an exact KEFT reproduction",\n            "original_qa_sha256": hashlib.sha256(args.alignment_qa.read_bytes()).hexdigest(),\n            "original_qa_rows": len(original_qa), "qa_preservation": "Verbatim parsed Q and A; no stripping or truncation",\n            "reference": "Original QA then rotated MCQ; LoRA disabled; eval; no_grad; decoder final normalized hidden vector at last prompt token",\n            "student": "MCQ only; LoRA enabled; same last-prompt-token position before Answer:<letter>",\n            "reference_vectors": "Computed on the fly, before each student forward; never cached",\n            "max_reference_length": args.max_reference_length,\n            "max_observed_reference_tokens": max(x["reference_tokens"] for x in alignment_lengths),\n            "validation": "MCQ only, unchanged LM loss and generation evaluation; no validation QA reference",\n        }\n        write_json(args.output / "metadata.json", metadata)\n    write_json(args.output / "length_exclusions.json", train_exclusions + val_exclusions)\n    with (args.output / "rotation_map.jsonl").open("w", encoding="utf-8") as handle:\n        for mapping in train_maps + val_maps:\n            handle.write(json.dumps(mapping) + "\\n")\n    print(f"Admitted: {len(train)} train questions -> {len(train_data)} presentations; {len(validation)} validation questions. Excluded groups: {len(train_exclusions) + len(val_exclusions)}.", flush=True)\n    print("Actual token/step budget:", json.dumps(metadata["budget"]), flush=True)\n    if args.prepare_only:\n        print("Preflight complete; no model was loaded or trained.", flush=True)\n        return\n\n    import torch\n    from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training\n    from transformers import AutoModelForCausalLM, BitsAndBytesConfig, Trainer, TrainingArguments\n    if not torch.cuda.is_available():\n        raise RuntimeError("Enable a Colab CUDA GPU. Tokenizer preflight is saved.")\n    test = read_csv(args.test, MCQ_FIELDS)\n    if any(r["Answer"].strip().lower() not in list(LETTERS) for r in test):\n        raise ValueError("Invalid test answer label")\n    if args.smoke_test:\n        test = test[:2]\n    # Include the assistant target in context checks; never truncate evaluation silently.\n    if any(tokenize_mcq(r, tokenizer, args.max_length)[0] is None for r in test):\n        raise ValueError("A test prompt exceeds max_length; increase the configured context before running")\n    quantization = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",\n                                   bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)\n    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=args.revision, quantization_config=quantization,\n                                                dtype=torch.float16, device_map={"": 0}, attn_implementation="sdpa")\n    revision = getattr(model.config, "_commit_hash", None)\n    if args.starting_adapter and revision != args.revision:\n        raise ValueError("Loaded base revision differs from pinned warm-start base")\n    if alignment_lengths and max(x["reference_tokens"] for x in alignment_lengths) > model.config.max_position_embeddings:\n        raise ValueError("A reference exceeds the model context window. No truncation or extra exclusions allowed.")\n    metadata.update(model_commit=revision, gpu=torch.cuda.get_device_name(0), gpu_count_used=1,\n                    test_rows=len(test), test_sha256=hashlib.sha256(args.test.read_bytes()).hexdigest(),\n                    versions={p: importlib.metadata.version(p) for p in\n                              ["torch", "transformers", "peft", "accelerate", "bitsandbytes"]},\n                    runtime={"python": platform.python_version(), "cuda": torch.version.cuda,\n                             "cudnn": torch.backends.cudnn.version(),\n                             "gpu_total_vram_gib": torch.cuda.get_device_properties(0).total_memory / 1024**3})\n    write_json(args.output / "metadata.json", metadata)\n    base_summary, base_predictions = None, None\n    if args.skip_base_evaluation:\n        print("Skipping baseline evaluation. Warm start follows." if args.starting_adapter else\n              "Skipping baseline evaluation. Starting fine-tuning from fresh base weights.", flush=True)\n    else:\n        base_summary, base_predictions = score(model, tokenizer, validation, test, args.output / "base", args.seed)\n    set_seed(args.seed)\n    model.train()\n    model.config.use_cache = False\n    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)\n    if args.starting_adapter:\n        model = load_trainable_adapter(model, args.starting_adapter)\n        metadata["warm_start_tensors_verified"] = True\n    else:\n        model = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,\n            bias="none", task_type="CAUSAL_LM",\n            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))\n        metadata["fresh_lora_zero_B_verified"] = all(bool((p == 0).all()) for n, p in model.named_parameters() if "lora_B" in n)\n        if not metadata["fresh_lora_zero_B_verified"]:\n            raise ValueError("Fresh LoRA B matrices must start at zero")\n    model.print_trainable_parameters()\n    trainable, total = model.get_nb_trainable_parameters()\n    metadata.update(trainable_parameters=trainable, total_parameters=total)\n    write_json(args.output / "metadata.json", metadata)\n    training_args = TrainingArguments(\n        output_dir=str(args.output / "training"), num_train_epochs=1, learning_rate=1e-4,\n        per_device_train_batch_size=1, per_device_eval_batch_size=1, gradient_accumulation_steps=16,\n        gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},\n        fp16=True, optim="paged_adamw_8bit", lr_scheduler_type="cosine", warmup_ratio=0.05,\n        weight_decay=0.01, max_grad_norm=0.3, logging_steps=1 if args.smoke_test else 5,\n        eval_strategy="no" if checkpoint_evaluations else "epoch", save_strategy="no",\n        save_only_model=bool(checkpoint_evaluations), prediction_loss_only=True, report_to="none",\n        seed=args.seed, data_seed=args.seed, dataloader_num_workers=0, remove_unused_columns=False,\n    )\n    objective = None\n    collator = partial(collate_examples, pad_token_id=tokenizer.pad_token_id)\n    callbacks = []\n    if checkpoint_evaluations:\n        from mcq_checkpointing import make_checkpoint_callback\n        callbacks.append(make_checkpoint_callback(validation, tokenizer, args.output,\n                                                  training_examples=len(train_data), parts=checkpoint_evaluations))\n    if args.starting_adapter or args.fresh_base:\n        from transformers import TrainerCallback\n\n        class WarmStartAudit(TrainerCallback):\n            checked = False\n\n            def on_pre_optimizer_step(self, args_, state, control, model=None, **kwargs):\n                if self.checked:\n                    return\n                grads = [p.grad for n, p in model.named_parameters() if "lora_" in n and p.grad is not None]\n                if not grads or not all(torch.isfinite(g).all() for g in grads) or not any(g.abs().max() > 0 for g in grads):\n                    raise FloatingPointError("LoRA gradients must be present, finite, and nonzero")\n                if any(p.requires_grad or p.grad is not None for n, p in model.named_parameters() if "lora_" not in n):\n                    raise RuntimeError("Base parameters must remain frozen")\n                write_json(args.output / "warm_start_gradient_check.json", {\n                    "lora_gradients_present_finite_nonzero": True, "frozen_base_has_no_gradients": True,\n                    "starting_adapter_sha256": metadata.get("starting_adapter_sha256"),\n                    "initialization": metadata["initialization"],\n                    "optimizer_step_before_update": state.global_step})\n                self.checked = True\n\n        callbacks.append(WarmStartAudit())\n    if original_qa is not None:\n        from mcq_alignment import AlignmentObjective, collate_alignment\n        from transformers import TrainerCallback\n        objective = AlignmentObjective(0.05)\n        collator = partial(collate_alignment, pad_token_id=tokenizer.pad_token_id)\n\n        class GradientAudit(TrainerCallback):\n            checked = False\n\n            def on_pre_optimizer_step(self, args_, state, control, model=None, **kwargs):\n                if self.checked:\n                    return\n                grads = [p.grad for n, p in model.named_parameters() if "lora_" in n and p.grad is not None]\n                if not grads or not all(torch.isfinite(g).all() for g in grads) or not any(g.abs().max() > 0 for g in grads):\n                    raise FloatingPointError("LoRA gradients must be present, finite, and nonzero")\n                if any(p.grad is not None for n, p in model.named_parameters() if "lora_" not in n):\n                    raise RuntimeError("Frozen base parameters received gradients")\n                write_json(args.output / "alignment_gradient_check.json", {\n                    "lora_gradients_present_finite_nonzero": True,\n                    "frozen_base_has_no_gradients": True,\n                    "reference_has_no_gradients": all(not x["reference_requires_grad"] and x["reference_grad_fn_is_none"] for x in objective.log),\n                    "optimizer_step_before_update": state.global_step,\n                })\n                self.checked = True\n\n        callbacks = [GradientAudit()]\n\n    class MCQTrainer(Trainer):\n        def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):\n            if objective is not None:\n                return objective(model, inputs, return_outputs, num_items_in_batch)\n            result = assistant_only_loss(model, inputs, return_outputs, num_items_in_batch)\n            loss = result[0] if return_outputs else result\n            if not torch.isfinite(loss).all():\n                raise FloatingPointError("Nonfinite MCQ loss")\n            return result\n\n    trainer = MCQTrainer(model=model, args=training_args, train_dataset=train_data, eval_dataset=val_data,\n                      processing_class=tokenizer, data_collator=collator, callbacks=callbacks)\n    trainer.model_accepts_loss_kwargs = True\n    gc.collect()\n    torch.cuda.empty_cache()\n    torch.cuda.reset_peak_memory_stats()\n    try:\n        trained = trainer.train()\n    finally:\n        if objective is not None:\n            write_json(args.output / "alignment_loss_log.json", objective.log)\n    write_json(args.output / "training_metrics.json", {**trained.metrics,\n               "peak_gpu_allocated_gib": torch.cuda.max_memory_allocated() / 1024**3})\n    write_json(args.output / "training_log.json", trainer.state.log_history)\n    adapter = args.output / "adapter"\n    model.save_pretrained(adapter, safe_serialization=True)\n    tokenizer.save_pretrained(adapter)\n    del trainer, model\n    gc.collect()\n    torch.cuda.empty_cache()\n    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=revision, quantization_config=quantization,\n                                                dtype=torch.float16, device_map={"": 0}, attn_implementation="sdpa")\n    model = PeftModel.from_pretrained(model, str(adapter), is_trainable=False)\n    tokenizer = AutoTokenizer.from_pretrained(adapter)\n    tuned_summary, tuned_predictions = score(model, tokenizer, validation, test, args.output / "finetuned", args.seed)\n    comparison = make_comparison(base_summary, base_predictions, tuned_summary, tuned_predictions, args.smoke_test)\n    write_json(args.output / "comparison.json", comparison)\n    print(f"Completed. Reloaded adapter and results: {args.output}", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')


In [ ]:
p = Path('mcq_augmentation.py')
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""Deterministic merge with explicit user acceptance policy and training checks."""\nimport argparse\nfrom collections import Counter\nimport csv\nimport hashlib\nimport json\nfrom pathlib import Path\n\nfrom finetune_qwen_mcq import LETTERS, MCQ_FIELDS, check_split_isolation, load_split, rotate_question, tokenize_mcq\n\n\ndef file_hash(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef candidate_hash(record):\n    return hashlib.sha256(json.dumps(record, sort_keys=True, ensure_ascii=False,\n                                     separators=(",", ":")).encode()).hexdigest()\n\n\ndef read_jsonl(path):\n    with Path(path).open(encoding="utf-8") as handle:\n        return [json.loads(line) for line in handle if line.strip()]\n\n\ndef identity(record):\n    variant_id = str(record["variant_id"])\n    # Normalize worker ID spellings without modifying generated question content.\n    if variant_id in ("v1", "v2"):\n        variant_id = variant_id[1:]\n    return int(record["source_row"]), int(variant_id)\n\n\ndef acceptance_policy(directory):\n    path = Path(directory) / "acceptance_policy.json"\n    if not path.exists():\n        return {"mode": "primary_review", "semantic_review_required": True}\n    policy = json.loads(path.read_text(encoding="utf-8"))\n    if policy.get("mode") == "as_generated":\n        if policy.get("user_authorized") is not True or not policy.get("authorization"):\n            raise ValueError("As-generated acceptance requires an explicit user instruction")\n    elif policy.get("mode") != "primary_review":\n        raise ValueError("Unknown acceptance policy")\n    return policy\n\n\ndef training_protocol(directory):\n    path = Path(directory) / "training_protocol.json"\n    if not path.exists():\n        return {"training_presentation_mode": "four_rotations", "presentations_per_mcq": 4,\n                "num_train_epochs": 1, "seed": 42, "checkpoint_evaluations": 0}\n    protocol = json.loads(path.read_text(encoding="utf-8"))\n    mode = protocol.get("training_presentation_mode")\n    if mode not in ("single_random", "four_rotations"):\n        raise ValueError("Unknown training presentation mode")\n    if protocol.get("presentations_per_mcq") != (1 if mode == "single_random" else 4):\n        raise ValueError("Presentation mode/count mismatch")\n    if protocol.get("num_train_epochs") != 1 or protocol.get("seed") != 42 or protocol.get("checkpoint_evaluations", 0) not in (0, 3):\n        raise ValueError("Invalid augmented training protocol")\n    return protocol\n\n\ndef candidate_technical_issues(record, packet, tokenizer=None, presentation_mode="four_rotations"):\n    """Check only data operability; no assessment of wording, evidence or answers."""\n    source_id, variant_id = identity(record)\n    if source_id != packet["source_row"] or variant_id not in (1, 2):\n        return ["Invalid source/variant identity"]\n    if any(not isinstance(record.get(k), str) or not record[k].strip() for k in ["Question", *LETTERS, "Answer"]):\n        return ["Missing MCQ fields"]\n    if len(record["Answer"]) != 1 or record["Answer"] not in LETTERS:\n        return ["Invalid answer label"]\n    if tokenizer is not None:\n        orders = rotate_question(candidate_row(record, packet))\n        if presentation_mode == "single_random":\n            orders = orders[:1]\n        if any(tokenize_mcq(v, tokenizer, 4096)[0] is None for v, _ in orders):\n            return ["A rotation exceeds 4096 tokens"]\n    return []\n\n\ndef approved_original(packet, repairs):\n    """Apply only explicit approved repairs; retain raw QA/MCQ evidence separately."""\n    original = dict(packet["original_mcq"])\n    seen = set()\n    evidence = {packet["source_row"]: packet["original_qa"],\n                **{int(r["source_row"]): r for r in packet["related_training_qa"]}}\n    for repair in repairs:\n        if repair["source_row"] != packet["source_row"]:\n            continue\n        field = repair["field"]\n        if field in seen or field not in [*LETTERS, "Question"] or repair.get("user_approved") is not True:\n            raise ValueError("Invalid, duplicate or unapproved original repair")\n        seen.add(field)\n        if original[field] != repair["before"]:\n            raise ValueError("Original repair does not match source bytes")\n        if field == original["Answer"] or field == "Question":\n            qa = evidence.get(repair.get("evidence_source_row"), {})\n            quote = repair.get("evidence_quote", "")\n            scope = "changes_question" if field == "Question" else "changes_correct_answer"\n            if not repair.get(scope) or not quote or not any(quote in qa.get(k, "") for k in ("Q", "A")):\n                raise ValueError("Original repair requires explicit approval scope and supplied evidence")\n        original[field] = repair["after"]\n    return original\n\n\ndef candidate_row(record, packet):\n    source_id, variant_id = identity(record)\n    return {**{k: record[k] for k in ["Question", *LETTERS, "Answer"]},\n            "Question": f"Regarding {packet[\'original_mcq\'][\'class\']} in this Unity game project:\\n{record[\'Question\']}",\n            "class": packet["original_mcq"]["class"], "source_row": source_id,\n            "split": "train", "variant_id": variant_id}\n\n\ndef candidate_issues(record, packet, tokenizer=None):\n    issues = []\n    source_id, variant_id = identity(record)\n    original = packet.get("approved_original_mcq", packet["original_mcq"])\n    if source_id != packet["source_row"] or variant_id not in (1, 2):\n        return ["Invalid source/variant identity"]\n    if any(not isinstance(record.get(k), str) or not record[k].strip() for k in ["Question", *LETTERS, "Answer"]):\n        return ["Missing MCQ fields"]\n    answer = record["Answer"]\n    if answer not in LETTERS or len(answer) != 1:\n        return ["Invalid answer label"]\n    if len({record[l].strip() for l in LETTERS}) != 4:\n        issues.append("Duplicate options")\n    if record[answer].strip().startswith("```") and any(not record[l].strip().startswith("```") for l in LETTERS):\n        issues.append("Code option format mismatch")\n    if record["Question"].strip().casefold() == original["Question"].strip().casefold():\n        issues.append("Question duplicates original")\n    kind = record.get("kind")\n    if kind not in ("same_answer", "focused") or (variant_id == 1 and kind != "same_answer"):\n        issues.append("Invalid variant kind")\n    if (variant_id == 1 or kind == "same_answer") and record[answer] != original[original["Answer"]]:\n        issues.append("Correct option not preserved verbatim")\n    old_distractors = {original[l].strip() for l in LETTERS if l != original["Answer"]}\n    if any(record[l].strip() in old_distractors for l in LETTERS if l != answer):\n        issues.append("Reused original distractor")\n    reasons = record.get("distractor_reasons", {})\n    if set(reasons) != set(LETTERS) - {answer} or any(not isinstance(x, str) or not x.strip() for x in reasons.values()):\n        issues.append("Missing false-option rationale")\n    if any("different method, type, event, or state behavior" in x for x in reasons.values() if isinstance(x, str)):\n        issues.append("Generic rationale placeholder")\n    evidence = {packet["source_row"]: packet["original_qa"],\n                **{int(r["source_row"]): r for r in packet["related_training_qa"]}}\n    quotes = record.get("evidence", [])\n    if not quotes:\n        issues.append("Missing supporting evidence")\n    for quote in quotes:\n        qa = evidence.get(int(quote.get("source_row", -1)))\n        text = quote.get("quote")\n        if qa is None or not isinstance(text, str) or not text.strip() or not any(text in qa[k] for k in ("Q", "A")):\n            issues.append("Evidence quote not verbatim in supplied QA")\n    if tokenizer is not None:\n        row = candidate_row(record, packet)\n        if any(tokenize_mcq(v, tokenizer, 4096)[0] is None for v, _ in rotate_question(row)):\n            issues.append("A rotation exceeds 4096 tokens")\n    return issues\n\n\ndef collect(directory):\n    directory = Path(directory)\n    manifest = json.loads((directory / "manifest.json").read_text())\n    packets, candidates = {}, {}\n    for worker in manifest["workers"]:\n        name = f"worker-{worker[\'worker\']}"\n        folder = directory / name\n        if file_hash(folder / "evidence.jsonl") != worker["evidence_sha256"]:\n            raise ValueError("Evidence packet changed")\n        worker_packets = read_jsonl(folder / "evidence.jsonl")\n        assigned = {p["source_row"] for p in worker_packets}\n        for packet in worker_packets:\n            sid = packet["source_row"]\n            if sid in packets:\n                raise ValueError("Duplicate evidence source")\n            packets[sid] = packet\n        for path in sorted(folder.glob("batch-*.jsonl")):\n            for record in read_jsonl(path):\n                key = identity(record)\n                if key in candidates:\n                    raise ValueError(f"Duplicate candidate ID {key}; archive replaced drafts explicitly")\n                if record.get("worker") != name or record.get("model") != "gpt-6-luna":\n                    raise ValueError("Invalid worker provenance")\n                if key[0] not in assigned:\n                    raise ValueError("Candidate source not assigned to worker")\n                candidates[key] = record\n    repairs_path = directory / "original_repairs.json"\n    repairs = json.loads(repairs_path.read_text()) if repairs_path.exists() else []\n    if any(r["source_row"] not in packets for r in repairs):\n        raise ValueError("Original repair targets an unadmitted source")\n    for packet in packets.values():\n        if any(r["source_row"] == packet["source_row"] for r in repairs):\n            packet["approved_original_mcq"] = approved_original(packet, repairs)\n    return manifest, packets, candidates\n\n\ndef merge(directory, output, tokenizer, validation, test, adapter, control):\n    directory, output = Path(directory), Path(output)\n    manifest, packets, candidates = collect(directory)\n    policy = acceptance_policy(directory)\n    protocol = training_protocol(directory)\n    orders = protocol["presentations_per_mcq"]\n    as_generated = policy["mode"] == "as_generated"\n    expected = {(sid, vid) for sid in packets for vid in (1, 2)}\n    if len(packets) != 1717 or set(candidates) != expected:\n        raise ValueError(f"Incomplete generation: {len(candidates)}/{2 * len(packets)} variants")\n    if not as_generated:\n        for path in directory.glob("worker-*/contradictions.jsonl"):\n            if any(not r.get("resolved", False) for r in read_jsonl(path)):\n                raise ValueError(f"Unresolved source contradiction: {path}")\n    decisions = {}\n    reviews_path = directory / "primary_reviews.jsonl"\n    if reviews_path.exists():\n        for row in read_jsonl(reviews_path):\n            decisions[(int(row["source_row"]), int(row["variant_id"]), row["candidate_sha256"])] = row\n    rows = []\n    repairs_path = directory / "original_repairs.json"\n    repairs = json.loads(repairs_path.read_text()) if repairs_path.exists() else []\n    repair_keys = set()\n    for repair in repairs:\n        key = (repair["source_row"], repair["field"])\n        if key in repair_keys or repair.get("user_approved") is not True or repair["field"] not in [*LETTERS, "Question"]:\n            raise ValueError("Invalid or unapproved original repair")\n        repair_keys.add(key)\n    for sid, packet in sorted(packets.items()):\n        original = {**approved_original(packet, repairs), "variant_id": 0}\n        if any(tokenize_mcq(v, tokenizer, 4096)[0] is None for v, _ in rotate_question(original)[:orders]):\n            raise ValueError(f"Original source {sid} no longer fits")\n        rows.append(original)\n        questions = {original["Question"].strip().casefold()}\n        for vid in (1, 2):\n            record = candidates[(sid, vid)]\n            if as_generated:\n                errors = candidate_technical_issues(record, packet, tokenizer, protocol["training_presentation_mode"])\n                if errors:\n                    raise ValueError(f"Variant {(sid, vid)} cannot be loaded for training: {errors}")\n            else:\n                errors = candidate_issues(record, packet, tokenizer)\n                review = decisions.get((sid, vid, candidate_hash(record)), {})\n                if errors or review.get("accepted") is not True or review.get("reviewer") != "primary" or not review.get("reason"):\n                    raise ValueError(f"Variant {(sid, vid)} not accepted: {errors or \'primary review required\'}")\n            question = record["Question"].strip().casefold()\n            if not as_generated and question in questions:\n                raise ValueError(f"Indistinct variants at source {sid}")\n            questions.add(question)\n            rows.append(candidate_row(record, packet))\n    heldout = load_split(validation, "validation")\n    check_split_isolation(rows, heldout)\n    for row in heldout:\n        if any(tokenize_mcq(v, tokenizer, 4096)[0] is None for v, _ in rotate_question(row)):\n            raise ValueError("Validation cohort no longer fits")\n    counts = Counter(v["Answer"] for row in rows for v, _ in rotate_question(row)[:orders])\n    if orders == 4 and counts != dict.fromkeys(LETTERS, 5151):\n        raise ValueError("Unbalanced presentations")\n    if sum(counts.values()) != len(rows) * orders:\n        raise ValueError("Presentation count mismatch")\n    output.mkdir(parents=True, exist_ok=False)\n    train_path = output / "train.csv"\n    with train_path.open("w", encoding="utf-8", newline="") as handle:\n        writer = csv.DictWriter(handle, fieldnames=[*MCQ_FIELDS, "source_row", "split", "variant_id"])\n        writer.writeheader()\n        writer.writerows({k: row[k] for k in writer.fieldnames} for row in rows)\n    control_data = json.loads(Path(control).read_text())\n    control_meta = control_data["metadata"]\n    if set(control_meta["training_source_rows"]) != set(packets) or control_meta.get("alignment"):\n        raise ValueError("Control source cohort mismatch or alignment present")\n    if file_hash(validation) != control_meta["input_sha256"]["validation"] or file_hash(test) != control_meta["test_sha256"]:\n        raise ValueError("Evaluation inputs changed from control")\n    result = {"status": "ready", "source_count": 1717, "mcqs": 5151, "presentations": 5151 * orders,\n              "training_presentation_mode": protocol["training_presentation_mode"],\n              "checkpoint_evaluations": protocol.get("checkpoint_evaluations", 0),\n              "training_protocol": protocol,\n              "training_protocol_sha256": file_hash(directory / "training_protocol.json") if (directory / "training_protocol.json").exists() else None,\n              "training_source_rows": sorted(packets),\n              "accepted_variants": 3434, "train_label_counts": dict(counts),\n              "base_revision": manifest["base_revision"],\n              "starting_adapter_sha256": manifest["starting_adapter_sha256"],\n              "adapter_config_sha256": file_hash(Path(adapter) / "adapter_config.json"),\n              "train_sha256": file_hash(train_path), "validation_sha256": file_hash(validation),\n              "test_sha256": file_hash(test), "control_sha256": file_hash(control),\n              "acceptance_policy": policy,\n              "acceptance_policy_sha256": file_hash(directory / "acceptance_policy.json") if (directory / "acceptance_policy.json").exists() else None,\n              "all_variants_semantically_reviewed": not as_generated,\n              "primary_reviews_sha256": file_hash(reviews_path) if reviews_path.exists() else None,\n              "original_repairs": repairs,\n              "generation_manifest": manifest,\n              "candidate_sha256": {f"{sid}:{vid}": candidate_hash(r) for (sid, vid), r in sorted(candidates.items())}}\n    if file_hash(Path(adapter) / "adapter_model.safetensors") != result["starting_adapter_sha256"]:\n        raise ValueError("Starting adapter differs from original MCQ control")\n    (output / "augmentation_manifest.json").write_text(json.dumps(result, indent=2) + "\\n", encoding="utf-8")\n    return result\n\n\ndef verify_training_manifest(path, train, validation, test, adapter, revision, fresh_base=False):\n    manifest = json.loads(Path(path).read_text(encoding="utf-8"))\n    mode = manifest.get("training_presentation_mode", "four_rotations")\n    if mode not in ("single_random", "four_rotations"):\n        raise ValueError("Unknown training presentation mode")\n    expected = {"status": "ready", "source_count": 1717, "mcqs": 5151,\n                "presentations": 5151 if mode == "single_random" else 20604, "accepted_variants": 3434}\n    if any(manifest.get(k) != v for k, v in expected.items()):\n        raise ValueError("Augmentation is incomplete or unreviewed")\n    if fresh_base and adapter is not None:\n        raise ValueError("Fresh-base training cannot load a starting adapter")\n    if (adapter is None and not fresh_base) or revision != manifest["base_revision"]:\n        raise ValueError("Original starting adapter and pinned base revision required")\n    policy = manifest.get("acceptance_policy", {})\n    if policy.get("mode") == "as_generated" and (policy.get("user_authorized") is not True or not policy.get("authorization")):\n        raise ValueError("As-generated acceptance requires an explicit user instruction")\n    for name, source in [("train", train), ("validation", validation), ("test", test)]:\n        if file_hash(source) != manifest[f"{name}_sha256"]:\n            raise ValueError(f"{name} input hash mismatch")\n    if not fresh_base:\n        for filename, key in [("adapter_model.safetensors", "starting_adapter_sha256"),\n                              ("adapter_config.json", "adapter_config_sha256")]:\n            if file_hash(Path(adapter) / filename) != manifest[key]:\n                raise ValueError("Starting adapter hash mismatch")\n        config = json.loads((Path(adapter) / "adapter_config.json").read_text())\n        if config["r"] != 16 or config["lora_alpha"] != 32 or config["lora_dropout"] != 0.05:\n            raise ValueError("Starting LoRA configuration differs")\n    rows = load_split(train, "train", allow_variants=True,\n                      validate_unique_options=manifest.get("acceptance_policy", {}).get("mode") != "as_generated")\n    groups = {}\n    for row in rows:\n        groups.setdefault(int(row["source_row"]), set()).add(int(row.get("variant_id", 0)))\n    if len(groups) != 1717 or set(groups) != set(manifest["training_source_rows"]) or any(v != {0, 1, 2} for v in groups.values()):\n        raise ValueError("Expected three variants per source")\n    return manifest\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--directory", type=Path, default=Path("data/mcq/augmentation"))\n    parser.add_argument("--output", type=Path)\n    parser.add_argument("--validation", type=Path, default=Path("data/mcq/generated-mcqs-final/validation.csv"))\n    parser.add_argument("--test", type=Path, default=Path("TestData.csv"))\n    parser.add_argument("--adapter", type=Path, default=Path("results/latest-run/adapter"))\n    parser.add_argument("--control", type=Path, default=Path("kaggle-alignment-bundle/data/control_results.json"))\n    parser.add_argument("--audit-output", type=Path, help="Persist resumable draft/check/review progress")\n    parser.add_argument("--tokenizer", type=Path, help="Optional local tokenizer for exact length audit")\n    args = parser.parse_args()\n    if args.output:\n        from transformers import AutoTokenizer\n        tokenizer = AutoTokenizer.from_pretrained(str(args.adapter), local_files_only=True)\n        print(json.dumps(merge(args.directory, args.output, tokenizer, args.validation, args.test, args.adapter, args.control), indent=2))\n    else:\n        _, packets, candidates = collect(args.directory)\n        policy = acceptance_policy(args.directory)\n        protocol = training_protocol(args.directory)\n        as_generated = policy["mode"] == "as_generated"\n        tokenizer = None\n        if args.tokenizer:\n            from transformers import AutoTokenizer\n            tokenizer = AutoTokenizer.from_pretrained(str(args.tokenizer), local_files_only=True)\n        decisions = {}\n        reviews = args.directory / "primary_reviews.jsonl"\n        if reviews.exists():\n            for row in read_jsonl(reviews):\n                decisions[(int(row["source_row"]), int(row["variant_id"]), row["candidate_sha256"])] = row\n        report = {"sources": len(packets), "draft_variants": len(candidates),\n                  "missing_variants": 3434 - len(candidates),\n                  "drafts_by_worker": dict(Counter(r["worker"] for r in candidates.values())),\n                  "acceptance_mode": policy["mode"],\n                  "accepted_current_variants": len(candidates) if as_generated else sum(decisions.get((*key, candidate_hash(row)), {}).get("accepted") is True\n                                                   for key, row in candidates.items()),\n                  "historically_primary_reviewed_variants": sum(decisions.get((*key, candidate_hash(row)), {}).get("accepted") is True\n                                                   for key, row in candidates.items())}\n        checker = (lambda r, p, t: candidate_technical_issues(r, p, t, protocol["training_presentation_mode"])) if as_generated else candidate_issues\n        report["technical_rejections" if as_generated else "structural_rejections"] = {\n            f"{sid}:{vid}": issues for (sid, vid), record in sorted(candidates.items())\n            if (issues := checker(record, packets[sid], tokenizer))}\n        if tokenizer is not None:\n            report["rotation_token_counts"] = {f"{sid}:{vid}": [tokenize_mcq(v, tokenizer, 4096)[1] for v, _ in\n                rotate_question(candidate_row(record, packets[sid]))] for (sid, vid), record in sorted(candidates.items())}\n        if args.audit_output:\n            args.audit_output.write_text(json.dumps(report, indent=2) + "\\n", encoding="utf-8")\n        print(json.dumps({k: v for k, v in report.items() if k != "rotation_token_counts"}, indent=2))\n', encoding='utf-8')


In [ ]:
p = Path('mcq_checkpointing.py')
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""Three adapter checkpoints and fixed-validation measurements within one epoch."""\nimport csv\nimport json\nimport math\nfrom pathlib import Path\n\n\ndef milestone_steps(max_steps, parts=3):\n    if max_steps < 1 or parts < 1:\n        raise ValueError("Positive optimizer step and checkpoint counts required")\n    return sorted({math.ceil(max_steps * i / parts) for i in range(1, parts + 1)})\n\n\ndef make_checkpoint_callback(validation, tokenizer, output, training_examples, parts=3, evaluate_fn=None):\n    import torch\n    from transformers import TrainerCallback\n    from finetune_qwen_coder import evaluate_mcq, write_json\n    if evaluate_fn is None:\n        evaluate_fn = evaluate_mcq\n    output = Path(output)\n\n    class ThirdEpochCheckpoints(TrainerCallback):\n        def __init__(self):\n            self.milestones = []\n            self.losses = {}\n            self.scores = []\n\n        def on_train_begin(self, args, state, control, **kwargs):\n            self.milestones = milestone_steps(state.max_steps, parts)\n            write_json(output / "checkpoint_schedule.json", {\n                "optimizer_steps": state.max_steps, "milestones": self.milestones,\n                "requested_checkpoints": parts, "actual_checkpoints": len(self.milestones),\n                "note": "Short smoke runs can have fewer than three optimizer updates."})\n\n        def on_step_end(self, args, state, control, **kwargs):\n            if state.global_step in self.milestones:\n                control.should_evaluate = True\n                control.should_save = True\n            return control\n\n        def on_evaluate(self, args, state, control, metrics=None, **kwargs):\n            if state.global_step in self.milestones:\n                loss = float(metrics["eval_loss"])\n                if not math.isfinite(loss):\n                    raise FloatingPointError("Checkpoint validation loss is not finite")\n                self.losses[state.global_step] = loss\n\n        def on_save(self, args, state, control, model=None, **kwargs):\n            step = state.global_step\n            if step not in self.milestones or any(r["optimizer_step"] == step for r in self.scores):\n                return\n            checkpoint = Path(args.output_dir) / f"checkpoint-{step}"\n            if not checkpoint.is_dir() or step not in self.losses:\n                raise RuntimeError("Checkpoint weights and validation loss must exist before accuracy scoring")\n            was_training = model.training\n            config = getattr(model, "config", None)\n            had_cache = config is not None and hasattr(config, "use_cache")\n            old_cache = config.use_cache if had_cache else None\n            devices = [torch.cuda.current_device()] if torch.cuda.is_available() else []\n            try:\n                with torch.random.fork_rng(devices=devices), torch.inference_mode():\n                    summary, _ = evaluate_fn(model, tokenizer, validation, checkpoint / "validation")\n            finally:\n                if had_cache:\n                    config.use_cache = old_cache\n                model.train(was_training)\n            accuracy = float(summary["accuracy"])\n            if not math.isfinite(accuracy):\n                raise FloatingPointError("Checkpoint validation accuracy is not finite")\n            losses = [r["loss"] for r in state.log_history if "loss" in r]\n            record = {"checkpoint_index": len(self.scores) + 1, "optimizer_step": step,\n                      "epoch": float(state.epoch),\n                      "training_examples_seen_estimate": min(training_examples, step * args.gradient_accumulation_steps * args.per_device_train_batch_size),\n                      "latest_logged_training_loss": float(losses[-1]) if losses else None,\n                      "validation_loss": self.losses[step], "validation_accuracy": accuracy,\n                      "validation_correct": summary["correct"], "validation_questions": summary["questions"],\n                      "checkpoint": str(checkpoint.relative_to(output))}\n            self.scores.append(record)\n            write_json(output / "checkpoint_scores.json", self.scores)\n            with (output / "checkpoint_scores.csv").open("w", encoding="utf-8", newline="") as handle:\n                writer = csv.DictWriter(handle, fieldnames=list(record))\n                writer.writeheader()\n                writer.writerows(self.scores)\n            print("Checkpoint validation:", json.dumps(record), flush=True)\n\n        def on_train_end(self, args, state, control, **kwargs):\n            if len(self.scores) != len(self.milestones):\n                raise RuntimeError("Incomplete checkpoint validation measurements")\n\n    return ThirdEpochCheckpoints()\n', encoding='utf-8')


In [ ]:
p = Path('.tools/build_kaggle_augmentation_bundle.py')
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""Build a separate private Kaggle continuation bundle after complete acceptance."""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport shutil\nimport sys\nimport zipfile\n\nROOT = Path(__file__).resolve().parents[1]\nsys.path.insert(0, str(ROOT))\nfrom mcq_augmentation import verify_training_manifest\n\n\ndef build(merged, output, owner, slug):\n    merged, output = Path(merged), Path(output)\n    validation = ROOT / "data/mcq/generated-mcqs-final/validation.csv"\n    test = ROOT / "TestData.csv"\n    adapter = ROOT / "results/latest-run/adapter"\n    manifest_path = merged / "augmentation_manifest.json"\n    manifest = json.loads(manifest_path.read_text())\n    verify_training_manifest(manifest_path, merged / "train.csv", validation, test, adapter, manifest["base_revision"])\n    output.mkdir(parents=True, exist_ok=False)\n    data = output / "dataset"\n    kernel = output / "notebook"\n    data.mkdir()\n    kernel.mkdir()\n    for source, name in [(merged / "train.csv", "train.csv"), (validation, "validation.csv"),\n                         (test, "TestData.csv"), (manifest_path, "augmentation_manifest.json"),\n                         (ROOT / "kaggle-alignment-bundle/data/control_results.json", "control_results.json")]:\n        shutil.copyfile(source, data / name)\n    for cohort in ("test", "validation_rotated"):\n        for filename in ("predictions.csv", "summary.json"):\n            source = ROOT / "results/latest-run/evaluation" / cohort / filename\n            if source.exists():\n                shutil.copyfile(source, data / f"control_{cohort}_{filename}")\n    if hashlib.sha256((data / "control_results.json").read_bytes()).hexdigest() != manifest["control_sha256"]:\n        raise ValueError("Control changed after augmentation merge")\n    shutil.copytree(adapter, data / "starting_adapter")\n    generation = ROOT / "data/mcq/augmentation"\n    for path in sorted(generation.rglob("*")):\n        if path.is_file() and path.suffix in (".json", ".jsonl", ".md"):\n            target = data / "generation_provenance" / path.relative_to(generation)\n            target.parent.mkdir(parents=True, exist_ok=True)\n            shutil.copyfile(path, target)\n    hashes = {p.relative_to(data).as_posix(): hashlib.sha256(p.read_bytes()).hexdigest()\n              for p in sorted(data.rglob("*")) if p.is_file()}\n    (data / "checksums.json").write_text(json.dumps(hashes, indent=2) + "\\n", encoding="utf-8")\n    scripts = ["benchmark.py", "finetune_qwen_coder.py", "finetune_qwen_mcq.py", "mcq_augmentation.py", "mcq_checkpointing.py",\n               ".tools/build_kaggle_augmentation_bundle.py",\n               "compare_alignment_control.py", "compare_augmentation_control.py", "tests/__init__.py",\n               "tests/test_mcq_rotation.py", "tests/test_mcq_loss.py", "tests/test_mcq_baseline.py",\n               "tests/test_mcq_augmentation.py", "tests/test_augmentation_comparison.py", "tests/test_mcq_checkpoints.py"]\n    cells = []\n    def cell(kind, source):\n        item = {"cell_type": kind, "id": f"augmentation-{len(cells):03d}", "metadata": {},\n                "source": source.splitlines(keepends=True)}\n        if kind == "code":\n            item.update(execution_count=None, outputs=[])\n            compile("\\n".join(line for line in source.splitlines() if not line.startswith("%")), item["id"], "exec")\n        cells.append(item)\n    cell("markdown", """# MCQ augmented continuation from the original adapter\n\nStarts from the original 78.07% test / 94.79% validation MCQ adapter, with **alignment disabled**.\n1,717 admitted sources × 3 MCQs × one shuffled option order = 5,151 presentations, one additional epoch.\nEvery MCQ is used once. Option order and training-example order are randomized deterministically.\nSave adapter weights and measure fixed validation loss/accuracy after each third: three checkpoints/scores.\nValidation and test remain unchanged. A new optimizer and scheduler are used independently for smoke/full runs.\n\nQLoRA: NF4, double quantization, fp16, rank 16, alpha 32, dropout .05; batch 1, accumulation 16,\nLR 1e-4, cosine, 5% warmup, weight decay .01, gradient clipping .3, answer-only loss, 4,096 tokens.\nThe base/tokenizer revision and input/checkpoint hashes are pinned. The GPU smoke exercises the longest\npresentation, gradients, frozen base weights, save/reload, and scoring before full training starts afresh.\n\nThis measures **additional training plus augmentation**. Without a matched additional-training control,\nany improvement cannot be attributed specifically to paraphrasing. Actual token and step budgets print before training.\nAcceptance follows the recorded user policy. When `as_generated`, variants are accepted as supplied\nwithout semantic review; technical input, cohort, rotation and length checks still run.\nAttach the private dataset, enable Internet and GPU, and Save Version → Save & Run All.\n""")\n    cell("code", """import os\nos.environ[\'CUDA_VISIBLE_DEVICES\'] = \'0\'\nos.environ[\'USE_TF\'] = \'0\'\nos.environ[\'TOKENIZERS_PARALLELISM\'] = \'false\'\nos.environ[\'PYTORCH_CUDA_ALLOC_CONF\'] = \'expandable_segments:True\'\n%pip install -q \'transformers==4.57.3\' \'peft==0.17.1\' \'accelerate==1.15.0\' \'bitsandbytes==0.48.2\' sentencepiece matplotlib\n""")\n    cell("code", """import hashlib, json, shutil, subprocess, sys, zipfile\nfrom pathlib import Path\nfrom datetime import datetime, timezone\nWORK = Path(\'/kaggle/working\')\nos.chdir(WORK)\nEXPECTED = """ + repr(hashes) + """\nmatches = sorted({p.parent for p in Path(\'/kaggle/input\').rglob(\'augmentation_manifest.json\')\n                  if all((p.parent / name).is_file() for name in EXPECTED)})\nif len(matches) != 1:\n    raise ValueError(f\'Attach exactly one matching augmented dataset, found {matches}\')\nDATA = matches[0]\nfor name, digest in EXPECTED.items():\n    if hashlib.sha256((DATA / name).read_bytes()).hexdigest() != digest:\n        raise ValueError(f\'Input hash mismatch: {name}\')\n    target = WORK / name\n    target.parent.mkdir(parents=True, exist_ok=True)\n    shutil.copyfile(DATA / name, target)\nADAPTER = WORK / \'starting_adapter\'\nif not (ADAPTER / \'adapter_model.safetensors\').is_file():\n    raise ValueError(\'Starting adapter folder missing from the attached dataset\')\nMANIFEST = json.loads(Path(\'augmentation_manifest.json\').read_text())\nREVISION = MANIFEST[\'base_revision\']\nRESULT = WORK / (\'mcq-augmentation-\' + datetime.now(timezone.utc).strftime(\'%Y%m%dT%H%M%S%fZ\'))\nRESULT.mkdir(exist_ok=False)\n""")\n    for name in scripts:\n        cell("code", f"p = Path({name!r})\\np.parent.mkdir(parents=True, exist_ok=True)\\np.write_text({(ROOT / name).read_text(encoding=\'utf-8\')!r}, encoding=\'utf-8\')\\n")\n    cell("code", """subprocess.run([sys.executable, \'-m\', \'unittest\', \'tests.test_mcq_augmentation\',\n                \'tests.test_mcq_rotation\', \'tests.test_mcq_loss\', \'tests.test_mcq_baseline\',\n                \'tests.test_augmentation_comparison\', \'tests.test_mcq_checkpoints\', \'-v\'], check=True)\nimport torch\nif not torch.cuda.is_available():\n    raise RuntimeError(\'Enable GPU\')\nprint(\'GPU:\', torch.cuda.get_device_name(0))\nfor name in """ + repr(scripts) + """:\n    target = RESULT / \'scripts\' / name\n    target.parent.mkdir(parents=True, exist_ok=True)\n    shutil.copyfile(name, target)\nfor name in [\'augmentation_manifest.json\', \'control_results.json\']:\n    shutil.copyfile(name, RESULT / name)\nif (WORK / \'generation_provenance\').is_dir():\n    shutil.copytree(WORK / \'generation_provenance\', RESULT / \'generation_provenance\')\nfor source in WORK.glob(\'control_*predictions.csv\'):\n    shutil.copyfile(source, RESULT / source.name)\nfor source in WORK.glob(\'control_*summary.json\'):\n    shutil.copyfile(source, RESULT / source.name)\ndef archive_results():\n    return shutil.make_archive(str(RESULT), \'zip\', RESULT)\ndef run(name, *flags):\n    command = [sys.executable, \'-u\', \'finetune_qwen_mcq.py\', \'--train\', \'train.csv\',\n               \'--validation\', \'validation.csv\', \'--test\', \'TestData.csv\', \'--output\', str(RESULT / name),\n               \'--starting-adapter\', str(ADAPTER), \'--augmentation-manifest\', \'augmentation_manifest.json\',\n               \'--revision\', REVISION, \'--skip-base-evaluation\', *flags]\n    try:\n        with (RESULT / (name + \'-console.log\')).open(\'w\', encoding=\'utf-8\') as log:\n            process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)\n            for line in process.stdout:\n                print(line, end=\'\', flush=True)\n                log.write(line)\n                log.flush()\n            if process.wait():\n                raise RuntimeError(f\'{name} failed; partial results archived\')\n    finally:\n        archive_results()\nrun(\'preflight\', \'--prepare-only\')\npreflight = json.loads((RESULT / \'preflight/metadata.json\').read_text())\nassert preflight[\'training_presentations\'] == MANIFEST[\'presentations\']\nassert preflight[\'train_label_counts\'] == MANIFEST[\'train_label_counts\']\nassert preflight[\'training_presentation_mode\'] == MANIFEST[\'training_presentation_mode\']\nassert len(preflight[\'training_source_rows\']) == 1717\nprint(\'Actual token and step budget:\', preflight[\'budget\'])\n""")\n    cell("markdown", "## Warm-start GPU smoke, then one additional full epoch\\n\\nThe full run reloads the original starting adapter independently; smoke weights and optimizer state are discarded.\\n")\n    cell("code", """import math\nrun(\'smoke\', \'--smoke-test\')\nsmoke = RESULT / \'smoke\'\naudit = json.loads((smoke / \'warm_start_gradient_check.json\').read_text())\nassert audit[\'lora_gradients_present_finite_nonzero\'] and audit[\'frozen_base_has_no_gradients\']\nassert json.loads((smoke / \'metadata.json\').read_text())[\'warm_start_tensors_verified\']\nassert json.loads((smoke / \'comparison.json\').read_text())[\'saved_adapter_reload_verified\']\nlogs = json.loads((smoke / \'training_log.json\').read_text())\nassert any(\'loss\' in row for row in logs)\nassert all(math.isfinite(row[key]) for row in logs for key in [\'loss\', \'eval_loss\'] if key in row)\nprint(\'GPU smoke passed. Reloading original adapter for full continuation.\')\nrun(\'continuation\')\nif MANIFEST.get(\'checkpoint_evaluations\'):\n    scores = json.loads((RESULT / \'continuation/checkpoint_scores.json\').read_text())\n    assert len(scores) == 3\n    for score in scores:\n        print(f"Checkpoint {score[\'checkpoint_index\']}: epoch={score[\'epoch\']:.3f}, validation loss={score[\'validation_loss\']:.4f}, accuracy={score[\'validation_accuracy\']:.2%}")\n""")\n    cell("code", """from compare_alignment_control import read_result\nfrom compare_augmentation_control import compare\ncomparison = compare(read_result(\'control_results.json\'), read_result(RESULT / \'continuation\'))\n(RESULT / \'validation_comparison.json\').write_text(json.dumps(comparison, indent=2) + \'\\\\n\')\nprint(json.dumps(comparison, indent=2))\nimport matplotlib.pyplot as plt\nlogs = json.loads((RESULT / \'continuation/training_log.json\').read_text())\nlosses = [r for r in logs if \'loss\' in r]\nplt.plot([r[\'step\'] for r in losses], [r[\'loss\'] for r in losses])\nplt.xlabel(\'Optimizer step\'); plt.ylabel(\'Answer-only training loss\')\nplt.savefig(RESULT / \'loss.png\', bbox_inches=\'tight\'); plt.show()\narchive_results()\n""")\n    cell("markdown", "## Interactive MCQ demo\\n\\nLoad the saved continuation adapter and use `demo(question, a, b, c, d)`. The final test and rotated validation outputs are retained under continuation/finetuned.\\n")\n    cell("code", """from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig\nfrom peft import PeftModel\nfrom finetune_qwen_coder import MODEL_ID\nfrom benchmark import build_prompt, parse_answer\ntokenizer = AutoTokenizer.from_pretrained(RESULT / \'continuation/adapter\')\nbase = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=REVISION, device_map={\'\': 0},\n    dtype=torch.float16, quantization_config=BitsAndBytesConfig(load_in_4bit=True,\n    bnb_4bit_quant_type=\'nf4\', bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16))\nmodel = PeftModel.from_pretrained(base, RESULT / \'continuation/adapter\').eval()\ndef demo(question, a, b, c, d):\n    row = dict(Question=question, a=a, b=b, c=c, d=d)\n    ids = tokenizer.apply_chat_template([{\'role\': \'user\', \'content\': build_prompt(row)}],\n        add_generation_prompt=True, return_tensors=\'pt\').to(model.device)\n    if ids.shape[-1] > 4096:\n        raise ValueError(\'Demo prompt exceeds reviewed context\')\n    with torch.inference_mode():\n        output = model.generate(ids, max_new_tokens=128, do_sample=False, pad_token_id=tokenizer.pad_token_id)\n    text = tokenizer.decode(output[0, ids.shape[-1]:], skip_special_tokens=True)\n    return {\'response\': text, \'prediction\': parse_answer(text)}\nimport csv\nwith Path(\'validation.csv\').open(encoding=\'utf-8-sig\', newline=\'\') as handle:\n    row = next(csv.DictReader(handle))\ndemo_check = demo(row[\'Question\'], row[\'a\'], row[\'b\'], row[\'c\'], row[\'d\'])\n(RESULT / \'demo_check.json\').write_text(json.dumps(demo_check, indent=2))\nprint(\'Reloaded demo response:\', demo_check)\nfrom IPython.display import FileLink, display\ndisplay(FileLink(archive_results()))\n""")\n    notebook = {"nbformat": 4, "nbformat_minor": 5, "cells": cells, "metadata": {\n        "kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"},\n        "language_info": {"name": "python"},\n        "kaggle": {"isInternetEnabled": True, "accelerator": "gpu", "language": "python", "sourceType": "notebook"}}}\n    notebook_name = "Qwen_MCQ_Augmented_Continuation.ipynb"\n    (kernel / notebook_name).write_text(json.dumps(notebook, indent=1) + "\\n", encoding="utf-8")\n    shutil.copyfile(kernel / notebook_name, data / notebook_name)\n    hashes[notebook_name] = hashlib.sha256((data / notebook_name).read_bytes()).hexdigest()\n    (data / "checksums.json").write_text(json.dumps(hashes, indent=2) + "\\n", encoding="utf-8")\n    dataset_id = f"{owner}/{slug}-data"\n    kernel_id = f"{owner}/{slug}"\n    (data / "dataset-metadata.json").write_text(json.dumps({"title": f"MCQ augmented continuation {slug}",\n        "id": dataset_id, "licenses": [{"name": "other"}]}, indent=2) + "\\n", encoding="utf-8")\n    (kernel / "kernel-metadata.json").write_text(json.dumps({"id": kernel_id,\n        "title": f"MCQ augmented continuation {slug}", "code_file": notebook_name,\n        "language": "python", "kernel_type": "notebook", "is_private": True,\n        "enable_gpu": True, "enable_internet": True, "dataset_sources": [dataset_id],\n        "competition_sources": [], "kernel_sources": []}, indent=2) + "\\n", encoding="utf-8")\n    (output / "README.md").write_text(f"""# Augmented MCQ continuation\n\nAll 3,434 variants were accepted under the recorded policy: `{manifest.get(\'acceptance_policy\', {}).get(\'mode\', \'primary_review\')}`.\nAs-generated acceptance does not assert that variants passed semantic review.\nExact control cohorts and adapter bytes are verified.\nDataset: `{dataset_id}`; notebook: `{kernel_id}`. Both must remain private.\n\n## Manual Kaggle setup\n\n1. Extract this bundle. Create a new **private Kaggle Dataset** and upload `dataset.zip`.\n   Kaggle should extract it so `train.csv`, `augmentation_manifest.json`, and the\n   `starting_adapter/` folder share one dataset root. Keep CSV bytes unchanged.\n2. Create a new notebook and use **File > Import Notebook** to import\n   `notebook/Qwen_MCQ_Augmented_Continuation.ipynb`.\n3. Add the private dataset as an input. Enable **Internet** and a **GPU** accelerator.\n   Start a fresh session. The notebook uses GPU 0 only.\n4. Use **Save Version > Save & Run All** when ready. No command-line publishing\n   or API key is required. Dataset and notebook metadata IDs are optional templates.\n5. Download the saved version\'s output ZIP. Three adapter checkpoints and\n   `continuation/checkpoint_scores.csv` / `.json` retain weights, validation loss,\n   and validation accuracy near one-third, two-thirds and the end of the epoch.\n   The final adapter is also under `continuation/adapter`.\n\nThe full training run uses one epoch over **5,151 shuffled presentations**, one\nshuffled option order per MCQ. It independently reloads the original starting\nadapter after the GPU smoke. Optimizer state continues across the three parts.\nValidation is measured at each checkpoint; test evaluation is at the end only.\nNo real 7B GPU run has been executed locally. The notebook runs its GPU smoke first.\n\nNotebook order: install pinned dependencies, verify inputs, CPU checks, tokenizer preflight,\nwarm-start longest-example GPU smoke, independent full continuation, saved adapter reload,\nfixed validation/test and rotated validation, paired control comparison, loss curve, demo, archive.\nActual tokens/steps and runtime versions are recorded. Smoke failure prevents full training.\nWithout a matched additional-training control, gains cannot be attributed specifically to paraphrasing.\n""", encoding="utf-8")\n    print(f"Built gated private bundle: {output}")\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--merged", type=Path, default=ROOT / "data/mcq/augmented-final")\n    parser.add_argument("--output", type=Path, default=ROOT / "kaggle-augmentation-bundle")\n    parser.add_argument("--owner", required=True)\n    parser.add_argument("--slug", required=True)\n    args = parser.parse_args()\n    build(args.merged, args.output, args.owner, args.slug)\n', encoding='utf-8')


In [ ]:
p = Path('compare_alignment_control.py')
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""Compare identical validation cohorts; never import unverified accuracy numbers."""\n\nimport argparse\nimport json\nfrom pathlib import Path\n\n\ndef read_result(path):\n    path = Path(path)\n    if path.is_file():\n        return json.loads(path.read_text(encoding="utf-8"))\n    import csv\n    metadata = json.loads((path / "metadata.json").read_text(encoding="utf-8"))\n    summary = json.loads((path / "finetuned/validation/summary.json").read_text(encoding="utf-8"))\n    with (path / "finetuned/validation/predictions.csv").open(encoding="utf-8-sig", newline="") as handle:\n        predictions = list(csv.DictReader(handle))\n    return {"metadata": metadata, "validation": summary, "predictions": predictions}\n\n\ndef compare(control, experiment):\n    a, b = control["metadata"], experiment["metadata"]\n    if a.get("alignment") is not None or not b.get("alignment"):\n        raise ValueError("Need a control without alignment and an experiment with alignment")\n    for key in ["model", "model_commit", "seed", "smoke_test", "admitted_questions",\n                "training_presentations", "train_label_counts", "input_sha256", "test_sha256", "settings"]:\n        if key not in a or key not in b or a[key] != b[key]:\n            raise ValueError(f"Control/experiment mismatch: {key}")\n    if sorted(a["training_source_rows"]) != sorted(b["training_source_rows"]):\n        raise ValueError("Training source cohorts differ")\n    pa, pb = control["predictions"], experiment["predictions"]\n    if len(pa) != len(pb) or not pa:\n        raise ValueError("Validation prediction cohorts differ")\n    for x, y in zip(pa, pb):\n        if any(str(x[k]) != str(y[k]) for k in ["row", "class", "question", "truth"]):\n            raise ValueError("Validation question order/content or answer key differs")\n    def correct(row):\n        return str(row["prediction"]).strip().lower() == str(row["truth"]).strip().lower()\n    ca, cb = sum(map(correct, pa)), sum(map(correct, pb))\n    for data, count in [(control, ca), (experiment, cb)]:\n        summary = data["validation"]\n        if summary["correct"] != count or summary["questions"] != len(pa) or abs(summary["accuracy"] - count/len(pa)) > 1e-10:\n            raise ValueError("Stored validation summary does not match raw predictions")\n    versions_differ = {k: {"control": a.get("versions", {}).get(k), "experiment": b.get("versions", {}).get(k)}\n                       for k in set(a.get("versions", {})) | set(b.get("versions", {}))\n                       if a.get("versions", {}).get(k) != b.get("versions", {}).get(k)}\n    return {"questions": len(pa), "control_correct": ca, "alignment_correct": cb,\n            "control_accuracy": ca/len(pa), "alignment_accuracy": cb/len(pa),\n            "accuracy_change_percentage_points": 100*(cb-ca)/len(pa),\n            "relative_accuracy_change": (cb-ca)/ca if ca else None,\n            "wrong_to_correct": sum(not correct(x) and correct(y) for x, y in zip(pa, pb)),\n            "correct_to_wrong": sum(correct(x) and not correct(y) for x, y in zip(pa, pb)),\n            "dependency_version_differences": versions_differ,\n            "same_gpu_model": a.get("gpu") == b.get("gpu"),\n            "note": "Single-seed validation comparison. Saved historical control is not a fresh paired rerun; inspect environment differences. No result establishes a KEFT reproduction or causal generalization benefit."}\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--control", type=Path, required=True)\n    parser.add_argument("--experiment", type=Path, required=True)\n    parser.add_argument("--output", type=Path, required=True)\n    args = parser.parse_args()\n    result = compare(read_result(args.control), read_result(args.experiment))\n    args.output.write_text(json.dumps(result, indent=2) + "\\n", encoding="utf-8")\n    print(json.dumps(result, indent=2))\n', encoding='utf-8')


In [ ]:
p = Path('compare_augmentation_control.py')
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""Compare continuation predictions with the unchanged historical validation control."""\nimport argparse\nimport json\nfrom pathlib import Path\n\nfrom compare_alignment_control import read_result\n\n\ndef compare(control, experiment):\n    a, b = control["metadata"], experiment["metadata"]\n    if a.get("alignment") or b.get("alignment") or not b.get("augmentation"):\n        raise ValueError("Expected original MCQ control and augmented continuation without alignment")\n    for key in ("model", "model_commit", "seed", "test_sha256"):\n        if a.get(key) != b.get(key):\n            raise ValueError(f"Control/continuation mismatch: {key}")\n    setting_changes = {k: {"control": a.get("settings", {}).get(k), "continuation": b.get("settings", {}).get(k)}\n                       for k in set(a.get("settings", {})) | set(b.get("settings", {}))\n                       if a.get("settings", {}).get(k) != b.get("settings", {}).get(k)}\n    if set(setting_changes) - {"rotations"}:\n        raise ValueError(f"Unexpected control/continuation setting changes: {setting_changes}")\n    if a["input_sha256"]["validation"] != b["input_sha256"]["validation"]:\n        raise ValueError("Validation bytes changed")\n    if set(a["training_source_rows"]) != set(b["training_source_rows"]):\n        raise ValueError("Training source cohort differs")\n    if a.get("smoke_test") or b.get("smoke_test"):\n        raise ValueError("Smoke results are not full comparisons")\n    pa, pb = control["predictions"], experiment["predictions"]\n    if len(pa) != len(pb) or len(pa) != 192:\n        raise ValueError("Validation cohort differs")\n    def correct(row):\n        return str(row["prediction"]).strip().lower() == str(row["truth"]).strip().lower()\n    changes = []\n    for x, y in zip(pa, pb):\n        if any(str(x[k]) != str(y[k]) for k in ("row", "class", "question", "truth")):\n            raise ValueError("Validation order/content/key differs")\n        if x["prediction"] != y["prediction"]:\n            changes.append({"row": x["row"], "control_prediction": x["prediction"],\n                            "continuation_prediction": y["prediction"], "truth": x["truth"],\n                            "control_correct": correct(x), "continuation_correct": correct(y)})\n    ca, cb = sum(map(correct, pa)), sum(map(correct, pb))\n    for result, count in ((control, ca), (experiment, cb)):\n        summary = result["validation"]\n        if summary["correct"] != count or summary["questions"] != len(pa) or abs(summary["accuracy"] - count / len(pa)) > 1e-10:\n            raise ValueError("Stored summary disagrees with predictions")\n    return {"questions": len(pa), "control_correct": ca, "continuation_correct": cb,\n            "control_accuracy": ca / len(pa), "continuation_accuracy": cb / len(pa),\n            "accuracy_change_percentage_points": 100 * (cb - ca) / len(pa),\n            "wrong_to_correct": sum(not correct(x) and correct(y) for x, y in zip(pa, pb)),\n            "correct_to_wrong": sum(correct(x) and not correct(y) for x, y in zip(pa, pb)),\n            "paired_prediction_changes": changes,\n            "training_settings_differences": setting_changes,\n            "dependency_versions": {"control": a.get("versions"), "continuation": b.get("versions")},\n            "gpu_models": {"control": a.get("gpu"), "continuation": b.get("gpu")},\n            "original_repairs": b["augmentation"].get("original_repairs", []),\n            "note": "Additional training plus augmentation, changed option-presentation budget and recorded original distractor repairs; without a matched additional-training control, gains cannot be attributed specifically to paraphrasing."}\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--control", type=Path, required=True)\n    parser.add_argument("--experiment", type=Path, required=True)\n    parser.add_argument("--output", type=Path, required=True)\n    args = parser.parse_args()\n    result = compare(read_result(args.control), read_result(args.experiment))\n    args.output.write_text(json.dumps(result, indent=2) + "\\n", encoding="utf-8")\n    print(json.dumps(result, indent=2))\n', encoding='utf-8')


In [ ]:
p = Path('tests/__init__.py')
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""Repository test package."""\n', encoding='utf-8')


In [ ]:
p = Path('tests/test_mcq_rotation.py')
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('import copy\nimport unittest\nfrom collections import Counter\n\nfrom finetune_qwen_mcq import check_split_isolation, prepare_split, rotate_question, rotation_diagnostics, tokenize_mcq, smoke_training_rows\n\n\nclass ChatTokenizer:\n    """Character tokenizer with an explicit assistant prefix and end token."""\n    def apply_chat_template(self, messages, tokenize, add_generation_prompt, return_dict=True):\n        text = "".join(f"<{m[\'role\']}>" + m[\'content\'] + "</turn>" for m in messages)\n        if add_generation_prompt:\n            text += "<assistant>"\n        ids = [ord(c) for c in text]\n        return {"input_ids": ids} if return_dict else ids\n\n\ndef example(source_id=1):\n    return {"source_row": str(source_id), "split": "train", "class": "Counter.cs",\n            "Question": f"Which statement is used in operation {source_id}?",\n            "a": "```csharp\\nCounter.instance\\n```", "b": "```csharp\\nCounter.Instance\\n```",\n            "c": \'return "a  b";\', "d": \'return "a b";\', "Answer": "b"}\n\n\nclass RotationTests(unittest.TestCase):\n    def test_single_random_uses_each_mcq_once_and_preserves_answer(self):\n        rows = [{**example(i // 3), "variant_id": str(i % 3)} for i in range(30)]\n        tokenizer = ChatTokenizer()\n        prepared, admitted, maps, exclusions = prepare_split(rows, tokenizer, 4096, 42, True, "single_random")\n        self.assertEqual(len(prepared), 30)\n        self.assertEqual(len(maps), 30)\n        self.assertFalse(exclusions)\n        self.assertEqual(len({(m["source_row"],m["variant_id"]) for m in maps}), 30)\n        for row, mapping in zip(admitted, maps):\n            expected, expected_map = rotate_question(row)[0]\n            self.assertEqual(mapping["new_to_original"], expected_map["new_to_original"])\n            self.assertEqual(expected[expected["Answer"]], row[row["Answer"]])\n        second, _, _, _ = prepare_split(rows, tokenizer, 4096, 42, True, "single_random")\n        self.assertEqual(prepared, second)\n        fixed, _, validation_maps, _ = prepare_split(rows, tokenizer, 4096, 42, False, "single_random")\n        self.assertEqual(len(fixed), 30)\n        self.assertEqual(len(validation_maps), 120)  # Evaluation diagnostics remain unchanged.\n\n    def test_every_option_visits_every_position_and_target_stays_correct(self):\n        for correct in "abcd":\n            row = {**example(), "Answer": correct}\n            before = copy.deepcopy(row)\n            variants = rotate_question(row, 42)\n            self.assertEqual(row, before)\n            self.assertEqual(Counter(v[\'Answer\'] for v, _ in variants), dict.fromkeys(\'abcd\', 1))\n            for letter in "abcd":\n                self.assertEqual({v[letter] for v, _ in variants}, {row[l] for l in \'abcd\'})\n            for v, mapping in variants:\n                self.assertEqual(v[v[\'Answer\']], row[correct])\n                self.assertEqual(v[\'Question\'], row[\'Question\'])\n                self.assertEqual(v[\'source_row\'], row[\'source_row\'])\n                for new, original in mapping[\'new_to_original\'].items():\n                    self.assertEqual(v[new], row[original])\n\n    def test_rotation_is_reproducible_independent_of_input_order(self):\n        rows = [example(i) for i in range(10)]\n        forward = {r[\'source_row\']: rotate_question(r, 7) for r in rows}\n        reverse = {r[\'source_row\']: rotate_question(r, 7) for r in reversed(rows)}\n        self.assertEqual(forward, reverse)\n        self.assertNotEqual(forward, {r[\'source_row\']: rotate_question(r, 8) for r in rows})\n\n    def test_smoke_includes_longest_fit_excludes_overlength_and_keeps_unique_ids(self):\n        rows = [example(i) for i in range(12)]\n        rows[10][\'Question\'] += \' context\' * 400\n        rows[11][\'Question\'] += \' context\' * 2000\n        selected = smoke_training_rows(rows, ChatTokenizer(), 10000, 42)\n        self.assertEqual(len(selected), 8)\n        self.assertEqual(selected[0][\'source_row\'], \'10\')\n        self.assertNotIn(\'11\', [r[\'source_row\'] for r in selected])\n        self.assertEqual(len({r[\'source_row\'] for r in selected}), 8)\n\n    def test_only_exact_assistant_answer_and_turn_end_receive_loss(self):\n        row = example()\n        item, _ = tokenize_mcq(row, ChatTokenizer(), 10000)\n        supervised = [token for token in item[\'labels\'] if token != -100]\n        self.assertEqual(\'\'.join(map(chr, supervised)), \'Answer:b</turn>\')\n        self.assertEqual(item[\'labels\'][-len(supervised):], supervised)\n        self.assertTrue(all(label == -100 for label in item[\'labels\'][:-len(supervised)]))\n        self.assertEqual(len(item[\'input_ids\']), len(item[\'labels\']))\n\n    def test_bad_template_prefix_fails_loudly(self):\n        class BadTokenizer(ChatTokenizer):\n            def apply_chat_template(self, messages, tokenize, add_generation_prompt, return_dict=True):\n                out = super().apply_chat_template(messages, tokenize, add_generation_prompt, return_dict=return_dict)\n                return [999] + out if not add_generation_prompt else out\n        with self.assertRaisesRegex(ValueError, \'prefix mismatch\'):\n            tokenize_mcq(example(), BadTokenizer(), 10000)\n\n    def test_group_length_rejection_preserves_fourfold_balance(self):\n        class OrderLengthTokenizer(ChatTokenizer):\n            def apply_chat_template(self, messages, tokenize, add_generation_prompt, return_dict=True):\n                out = super().apply_chat_template(messages, tokenize, add_generation_prompt, return_dict=return_dict)\n                # Exactly one order of source 2 becomes overlength.\n                if \'operation 2?\' in messages[0][\'content\'] and \'a) ```csharp\\nCounter.instance\' in messages[0][\'content\']:\n                    out = [1] * 20000 + out\n                return out\n        prepared, admitted, maps, exclusions = prepare_split([example(1), example(2)], OrderLengthTokenizer(), 10000, 42, True)\n        self.assertEqual(len(prepared), 4)\n        self.assertEqual([r[\'source_row\'] for r in admitted], [\'1\'])\n        self.assertEqual([r[\'source_row\'] for r in exclusions], [2])\n        self.assertEqual(Counter(m[\'answer\'] for m in maps), dict.fromkeys(\'abcd\', 1))\n\n    def test_fixed_validation_and_no_split_overlap(self):\n        row = {**example(2), \'split\': \'validation\'}\n        check_split_isolation([example()], [row])\n        prepared, admitted, maps, exclusions = prepare_split([row], ChatTokenizer(), 10000, 42, False)\n        expected, _ = tokenize_mcq(row, ChatTokenizer(), 10000)\n        self.assertEqual(prepared, [expected])\n        self.assertEqual(admitted, [row])\n        self.assertEqual(len(maps), 4)\n        self.assertFalse(exclusions)\n        with self.assertRaisesRegex(ValueError, \'Source ids overlap\'):\n            check_split_isolation([example()], [{**row, \'source_row\': \'1\'}])\n        with self.assertRaisesRegex(ValueError, \'questions cross splits\'):\n            check_split_isolation([example()], [{**row, \'Question\': example()[\'Question\'].upper()}])\n\n    def test_diagnostic_maps_predictions_back_to_original_answer_text(self):\n        rows = [example(1), example(2), example(3)]\n        predictions = []\n        for index, row in enumerate(rows):\n            for variant, mapping in rotate_question(row, 42):\n                # First always correct, second always same wrong text, third always invalid.\n                original = row[\'Answer\'] if index == 0 else \'a\'\n                predicted = next(l for l, old in mapping[\'new_to_original\'].items() if old == original) if index < 2 else None\n                predictions.append({\'prediction\': predicted, \'correct\': predicted == variant[\'Answer\']})\n        out = rotation_diagnostics(rows, predictions, 42)\n        self.assertAlmostEqual(out[\'consistent_answer_text_rate\'], 2 / 3)\n        self.assertAlmostEqual(out[\'all_four_correct_rate\'], 1 / 3)\n        for stats in out[\'accuracy_by_correct_position\'].values():\n            self.assertEqual(stats[\'questions\'], 3)\n            self.assertEqual(stats[\'correct\'], 1)\n\n\nif __name__ == \'__main__\':\n    unittest.main()\n', encoding='utf-8')


In [ ]:
p = Path('tests/test_mcq_loss.py')
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""Check loss/gradient equivalence on a real, randomly initialized tiny Qwen."""\n\nimport unittest\n\n\nclass MCQLossTests(unittest.TestCase):\n    @classmethod\n    def setUpClass(cls):\n        import torch\n        from transformers import Qwen2Config, Qwen2ForCausalLM\n        cls.torch = torch\n        torch.set_num_threads(2)\n        torch.manual_seed(42)\n        cls.model = Qwen2ForCausalLM(Qwen2Config(\n            vocab_size=101, hidden_size=32, intermediate_size=64,\n            num_hidden_layers=1, num_attention_heads=4, num_key_value_heads=2,\n            max_position_embeddings=128, attention_dropout=0.0,\n        )).eval()\n\n    def test_loss_and_all_parameter_gradients_match_dense_masked_causal_loss(self):\n        from finetune_qwen_mcq import assistant_only_loss\n        t = self.torch\n        ids = t.randint(0, 101, (2, 24))\n        labels = ids.clone()\n        labels[0, :18] = -100\n        labels[1, :15] = -100\n        labels[1, 21:] = -100\n        mask = t.ones_like(ids)\n        mask[1, 21:] = 0\n        batch = {"input_ids": ids, "attention_mask": mask, "labels": labels}\n        for denominator in [None, t.tensor(24)]:\n            self.model.zero_grad(set_to_none=True)\n            output = self.model(input_ids=ids, attention_mask=mask, use_cache=False)\n            full_loss = t.nn.functional.cross_entropy(\n                output.logits[:, :-1].reshape(-1, 101), labels[:, 1:].reshape(-1),\n                ignore_index=-100, reduction="mean" if denominator is None else "sum")\n            if denominator is not None:\n                full_loss = full_loss / denominator\n            full_loss.backward()\n            gradients = {n: p.grad.clone() for n, p in self.model.named_parameters() if p.grad is not None}\n            self.model.zero_grad(set_to_none=True)\n            small_loss, reduced = assistant_only_loss(self.model, batch, True, denominator)\n            t.testing.assert_close(small_loss, full_loss)\n            small_loss.backward()\n            for name, p in self.model.named_parameters():\n                if name in gradients:\n                    t.testing.assert_close(p.grad, gradients[name], rtol=2e-4, atol=2e-6)\n            self.assertLess(reduced.logits.shape[1], ids.shape[1])\n            self.assertIn("labels", batch)  # Loss must not mutate the collator batch.\n\n    def test_no_supervised_targets_rejected(self):\n        from finetune_qwen_mcq import assistant_only_loss\n        t = self.torch\n        batch = {"input_ids": t.ones((1, 8), dtype=t.long),\n                 "labels": t.full((1, 8), -100, dtype=t.long)}\n        with self.assertRaisesRegex(ValueError, "no supervised"):\n            assistant_only_loss(self.model, batch)\n\n\nif __name__ == "__main__":\n    unittest.main()\n', encoding='utf-8')


In [ ]:
p = Path('tests/test_mcq_baseline.py')
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('import copy\nimport unittest\n\nfrom finetune_qwen_mcq import make_comparison\n\n\nclass BaselineTests(unittest.TestCase):\n    def test_skipped_baseline_has_no_invented_metrics(self):\n        tuned = {"test": {"accuracy": 0.75}}\n        result = make_comparison(None, None, tuned, [{"correct": True}])\n        self.assertIsNone(result["base"])\n        self.assertEqual(result["baseline_evaluation"], "skipped")\n        for key in ["test_accuracy_change_percentage_points", "wrong_to_correct", "correct_to_wrong"]:\n            self.assertIsNone(result[key])\n        self.assertEqual(result["finetuned"], tuned)\n\n    def test_measured_baseline_retains_delta_and_prediction_flips(self):\n        base, tuned = {"test": {"accuracy": 0.5}}, {"test": {"accuracy": 0.75}}\n        before = copy.deepcopy((base, tuned))\n        old = [{"correct": x} for x in [False, True, False, True]]\n        new = [{"correct": x} for x in [True, False, True, True]]\n        result = make_comparison(base, old, tuned, new, True)\n        self.assertEqual(result["test_accuracy_change_percentage_points"], 25)\n        self.assertEqual(result["wrong_to_correct"], 2)\n        self.assertEqual(result["correct_to_wrong"], 1)\n        self.assertEqual(result["baseline_evaluation"], "measured_this_run")\n        self.assertTrue(result["smoke_test"])\n        self.assertEqual((base, tuned), before)\n\n\nif __name__ == "__main__":\n    unittest.main()\n', encoding='utf-8')


In [ ]:
p = Path('tests/test_mcq_augmentation.py')
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('import copy\nimport csv\nimport json\nimport importlib.util\nfrom pathlib import Path\nimport tempfile\nimport shutil\nimport unittest\nfrom unittest.mock import patch\n\nfrom finetune_qwen_mcq import load_split, rotate_question, smoke_training_rows\nfrom mcq_augmentation import approved_original, candidate_hash, candidate_issues, file_hash, merge, verify_training_manifest\nfrom tests.test_mcq_rotation import ChatTokenizer, example\n\n\nclass AugmentationTests(unittest.TestCase):\n    def test_correct_option_repair_requires_explicit_scope_and_training_evidence(self):\n        original = example(1)\n        packet = {"source_row": 1, "original_mcq": original, "original_qa": {"Q": "q", "A": "bad annotation"},\n                  "related_training_qa": [{"source_row": 2, "Q": "q", "A": "Supported corrected fact."}]}\n        repair = {"source_row": 1, "field": "b", "before": original["b"], "after": "Supported corrected fact.",\n                  "user_approved": True, "evidence_source_row": 2, "evidence_quote": "Supported corrected fact."}\n        with self.assertRaisesRegex(ValueError, "scope"):\n            approved_original(packet, [repair])\n        repair["changes_correct_answer"] = True\n        self.assertEqual(approved_original(packet, [repair])["b"], repair["after"])\n        self.assertEqual(packet["original_mcq"], original)\n        repair["evidence_source_row"] = 99\n        with self.assertRaisesRegex(ValueError, "evidence"):\n            approved_original(packet, [repair])\n        question_repair = {**repair, "field": "Question", "before": original["Question"],\n                           "after": "Which supported fact is stated?", "evidence_source_row": 2}\n        with self.assertRaisesRegex(ValueError, "scope"):\n            approved_original(packet, [question_repair])\n        question_repair["changes_question"] = True\n        self.assertEqual(approved_original(packet, [question_repair])["Question"], question_repair["after"])\n\n    def test_variant_zero_preserves_original_permutations(self):\n        original = example()\n        self.assertEqual([m["new_to_original"] for _, m in rotate_question(original)],\n                         [m["new_to_original"] for _, m in rotate_question({**original, "variant_id": "0"})])\n        for vid in (1, 2):\n            row = {**original, "variant_id": str(vid)}\n            rotations = rotate_question(row)\n            self.assertEqual({r["Answer"] for r, _ in rotations}, set("abcd"))\n            self.assertEqual(rotations, rotate_question(row))\n            self.assertTrue(all(r[r["Answer"]] == original[original["Answer"]] for r, _ in rotations))\n            self.assertTrue(all(m["variant_id"] == vid for _, m in rotations))\n\n    def test_duplicate_identity_allowed_only_for_distinct_training_variants(self):\n        with tempfile.TemporaryDirectory() as folder:\n            path = Path(folder) / "data.csv"\n            rows = [{**example(), "variant_id": str(v)} for v in (0, 1, 2)]\n            def write(data):\n                with path.open("w", newline="", encoding="utf-8") as handle:\n                    writer = csv.DictWriter(handle, fieldnames=list(rows[0]))\n                    writer.writeheader()\n                    writer.writerows(data)\n            write(rows)\n            self.assertEqual(len(load_split(path, "train", True)), 3)\n            with self.assertRaises(ValueError):\n                load_split(path, "train")\n            write(rows + [rows[1]])\n            with self.assertRaisesRegex(ValueError, "repeated"):\n                load_split(path, "train", True)\n\n    def test_smoke_selects_longest_variant_not_first_source_match(self):\n        rows = [{**example(1), "variant_id": str(v)} for v in (0, 1, 2)]\n        rows[2]["Question"] += " long" * 100\n        selected = smoke_training_rows(rows, ChatTokenizer(), 4096, 42)\n        self.assertEqual(selected[0]["variant_id"], "2")\n\n    def test_review_structural_checks_and_content_bound_hash(self):\n        original = example(1)\n        packet = {"source_row": 1, "original_mcq": original,\n                  "original_qa": {"Q": "Which counter?", "A": original["b"]}, "related_training_qa": []}\n        record = {"source_row": 1, "variant_id": 1, "Question": "Which exact Counter field is accessed?",\n                  "a": original["b"], "b": "```csharp\\nCounter.Active\\n```", "c": "```csharp\\nCounter.Current\\n```", "d": "```csharp\\nCounter.Global\\n```",\n                  "Answer": "a", "kind": "same_answer", "evidence": [{"source_row": 1, "quote": original["b"]}],\n                  "distractor_reasons": dict.fromkeys("bcd", "The supplied expression names Instance instead.")}\n        self.assertEqual(candidate_issues(record, packet, ChatTokenizer()), [])\n        changed = copy.deepcopy(record)\n        changed["a"] = changed["a"].lower()\n        self.assertIn("Correct option not preserved verbatim", candidate_issues(changed, packet))\n        self.assertNotEqual(candidate_hash(record), candidate_hash(changed))\n        self.assertEqual(candidate_hash(record), candidate_hash(dict(reversed(list(record.items())))))\n        changed = copy.deepcopy(record)\n        changed["evidence"][0]["source_row"] = 99\n        self.assertIn("Evidence quote not verbatim in supplied QA", candidate_issues(changed, packet))\n        changed = {**record, "Question": "x" * 5000}\n        self.assertIn("A rotation exceeds 4096 tokens", candidate_issues(changed, packet, ChatTokenizer()))\n\n    def test_unreviewed_manifest_cannot_start_training(self):\n        with tempfile.TemporaryDirectory() as folder:\n            path = Path(folder) / "manifest.json"\n            path.write_text(json.dumps({"status": "generation_pending"}))\n            with self.assertRaisesRegex(ValueError, "incomplete or unreviewed"):\n                verify_training_manifest(path, None, None, None, None, None)\n\n    def test_complete_merge_is_deterministic_and_requires_content_bound_reviews(self):\n        with tempfile.TemporaryDirectory() as folder:\n            directory = Path(folder)\n            packets, candidates = {}, {}\n            for sid in range(1717):\n                original = {**example(sid), "a": "correct", "b": "wrong0", "c": "wrong1", "d": "wrong2", "Answer": "a"}\n                packets[sid] = {"source_row": sid, "original_mcq": original,\n                                "original_qa": {"Q": original["Question"], "A": "correct"}, "related_training_qa": []}\n                for vid in (1, 2):\n                    candidates[(sid, vid)] = {"source_row": sid, "variant_id": vid, "Question": f"Alternative {vid} for source {sid}?",\n                        "a": "correct", "b": "false3", "c": "false4", "d": "false5", "Answer": "a", "kind": "same_answer",\n                        "evidence": [{"source_row": sid, "quote": "correct"}], "distractor_reasons": dict.fromkeys("bcd", "Different literal.")}\n            adapter = directory / "adapter"\n            adapter.mkdir()\n            (adapter / "adapter_model.safetensors").write_bytes(b"fixture-adapter")\n            (adapter / "adapter_config.json").write_text(json.dumps({"r": 16, "lora_alpha": 32, "lora_dropout": 0.05}))\n            validation, test = directory / "validation.csv", directory / "test.csv"\n            test.write_text("unchanged-test")\n            val_row = {**example(2000), "split": "validation"}\n            with validation.open("w", newline="") as handle:\n                writer = csv.DictWriter(handle, fieldnames=list(val_row))\n                writer.writeheader()\n                writer.writerow(val_row)\n            control = directory / "control.json"\n            control.write_text(json.dumps({"metadata": {"training_source_rows": list(packets),\n                "input_sha256": {"validation": file_hash(validation)}, "test_sha256": file_hash(test)}}))\n            reviews = directory / "primary_reviews.jsonl"\n            reviews.write_text("".join(json.dumps({"source_row": sid, "variant_id": vid, "accepted": True,\n                "reviewer": "primary", "reason": "Synthetic fixture decision", "candidate_sha256": candidate_hash(row)}) + "\\n"\n                for (sid, vid), row in candidates.items()))\n            manifest = {"base_revision": "pinned", "starting_adapter_sha256": file_hash(adapter / "adapter_model.safetensors")}\n            def invoke(name, candidate_dict):\n                with patch("mcq_augmentation.collect", return_value=(manifest, packets, candidate_dict)):\n                    return merge(directory, directory / name, ChatTokenizer(), validation, test, adapter, control)\n            first = invoke("first", candidates)\n            second = invoke("second", dict(reversed(list(candidates.items()))))\n            self.assertEqual(first, second)\n            self.assertEqual((directory / "first/train.csv").read_bytes(), (directory / "second/train.csv").read_bytes())\n            self.assertEqual(first["train_label_counts"], dict.fromkeys("abcd", 5151))\n            verified = verify_training_manifest(directory / "first/augmentation_manifest.json", directory / "first/train.csv",\n                                                validation, test, adapter, "pinned")\n            self.assertEqual(verified["presentations"], 20604)\n            # Exercise notebook authoring with a tiny checkpoint fixture, no GPU/network.\n            real_root = Path(__file__).resolve().parents[1]\n            spec = importlib.util.spec_from_file_location("augmentation_builder", real_root / ".tools/build_kaggle_augmentation_bundle.py")\n            builder = importlib.util.module_from_spec(spec)\n            spec.loader.exec_module(builder)\n            fixture_root = directory / "bundle-root"\n            (fixture_root / "data/mcq/generated-mcqs-final").mkdir(parents=True)\n            shutil.copyfile(validation, fixture_root / "data/mcq/generated-mcqs-final/validation.csv")\n            shutil.copyfile(test, fixture_root / "TestData.csv")\n            shutil.copytree(adapter, fixture_root / "results/latest-run/adapter")\n            (fixture_root / "kaggle-alignment-bundle/data").mkdir(parents=True)\n            shutil.copyfile(control, fixture_root / "kaggle-alignment-bundle/data/control_results.json")\n            (fixture_root / "data/mcq/augmentation").mkdir()\n            for source in [*real_root.glob("*.py"), *real_root.glob("tests/*.py")]:\n                target = fixture_root / source.relative_to(real_root)\n                target.parent.mkdir(exist_ok=True, parents=True)\n                shutil.copyfile(source, target)\n            (fixture_root / ".tools").mkdir()\n            shutil.copyfile(real_root / ".tools/build_kaggle_augmentation_bundle.py",\n                            fixture_root / ".tools/build_kaggle_augmentation_bundle.py")\n            with patch.object(builder, "ROOT", fixture_root):\n                builder.build(directory / "first", directory / "bundle", "fixture", "unit-test")\n            notebook_path = directory / "bundle/notebook/Qwen_MCQ_Augmented_Continuation.ipynb"\n            notebook = json.loads(notebook_path.read_text())\n            for cell in notebook["cells"]:\n                if cell["cell_type"] == "code":\n                    compile("\\n".join(line for line in "".join(cell["source"]).splitlines() if not line.startswith("%")), cell["id"], "exec")\n            kernel_meta = json.loads((directory / "bundle/notebook/kernel-metadata.json").read_text())\n            self.assertTrue(kernel_meta["is_private"])\n            self.assertEqual(kernel_meta["dataset_sources"], ["fixture/unit-test-data"])\n            changed = copy.deepcopy(candidates)\n            changed[(0, 1)]["Question"] += " changed"\n            with self.assertRaisesRegex(ValueError, "not accepted"):\n                invoke("changed", changed)\n            with self.assertRaisesRegex(ValueError, "Incomplete"):\n                invoke("missing", {k: v for k, v in candidates.items() if k != (0, 1)})\n            with self.assertRaisesRegex(ValueError, "revision"):\n                verify_training_manifest(directory / "first/augmentation_manifest.json", directory / "first/train.csv",\n                                         validation, test, adapter, "wrong-pin")\n            # User acceptance must bypass semantic/review gates without changing content.\n            (directory / "acceptance_policy.json").write_text(json.dumps({\n                "mode": "as_generated", "user_authorized": True,\n                "authorization": "Accept generated variants as supplied; no reviews"}))\n            supplied = copy.deepcopy(candidates)\n            supplied[(0, 1)].update(Question=packets[0]["original_mcq"]["Question"],\n                                    b="correct", evidence=[], distractor_reasons={})\n            with patch("mcq_augmentation.candidate_issues", side_effect=AssertionError("Review gate must not run")):\n                accepted = invoke("as-generated", supplied)\n            self.assertFalse(accepted["all_variants_semantically_reviewed"])\n            self.assertEqual(accepted["acceptance_policy"]["mode"], "as_generated")\n            verify_training_manifest(directory / "as-generated/augmentation_manifest.json",\n                                     directory / "as-generated/train.csv", validation, test, adapter, "pinned")\n            with (directory / "as-generated/train.csv").open(newline="", encoding="utf-8") as handle:\n                unchanged_variant = list(csv.DictReader(handle))[1]\n            self.assertEqual(unchanged_variant["a"], unchanged_variant["b"])\n            bad_schema = copy.deepcopy(supplied)\n            bad_schema[(0, 1)]["Question"] = ""\n            with self.assertRaisesRegex(ValueError, "Missing MCQ fields"):\n                invoke("bad-schema", bad_schema)\n            too_long = copy.deepcopy(supplied)\n            too_long[(0, 1)]["Question"] = "x" * 5000\n            with self.assertRaisesRegex(ValueError, "4096 tokens"):\n                invoke("too-long", too_long)\n            (directory / "training_protocol.json").write_text(json.dumps({\n                "training_presentation_mode":"single_random", "presentations_per_mcq":1,\n                "num_train_epochs":1, "seed":42, "checkpoint_evaluations":3}))\n            single = invoke("single-random", supplied)\n            self.assertEqual(single["presentations"], 5151)\n            self.assertEqual(single["checkpoint_evaluations"], 3)\n            self.assertEqual(sum(single["train_label_counts"].values()), 5151)\n            single_rows = load_split(directory / "single-random/train.csv", "train", True, False)\n            from finetune_qwen_mcq import prepare_split\n            prepared, _, maps, _ = prepare_split(single_rows, ChatTokenizer(), 4096, 42, True, "single_random")\n            self.assertEqual(len(prepared), 5151)\n            from collections import Counter\n            self.assertEqual(dict(Counter(m["answer"] for m in maps)), single["train_label_counts"])\n            verify_training_manifest(directory / "single-random/augmentation_manifest.json",\n                                     directory / "single-random/train.csv", validation, test, adapter, "pinned")\n            (directory / "acceptance_policy.json").write_text(json.dumps({"mode":"as_generated"}))\n            with self.assertRaisesRegex(ValueError, "explicit user"):\n                invoke("not-authorized", supplied)\n            (adapter / "adapter_model.safetensors").write_bytes(b"changed")\n            with self.assertRaisesRegex(ValueError, "adapter hash"):\n                verify_training_manifest(directory / "first/augmentation_manifest.json", directory / "first/train.csv",\n                                         validation, test, adapter, "pinned")\n\n\nclass WarmStartTests(unittest.TestCase):\n    def test_real_peft_warm_load_backward_and_reload(self):\n        import torch\n        from peft import LoraConfig, PeftModel, get_peft_model\n        from transformers import Qwen2Config, Qwen2ForCausalLM\n        from finetune_qwen_mcq import assistant_only_loss, load_trainable_adapter\n        torch.set_num_threads(2)\n        torch.manual_seed(42)\n        config = Qwen2Config(vocab_size=101, hidden_size=32, intermediate_size=64,\n                             num_hidden_layers=1, num_attention_heads=4, num_key_value_heads=2)\n        base = Qwen2ForCausalLM(config)\n        state = copy.deepcopy(base.state_dict())\n        model = get_peft_model(base, LoraConfig(r=4, lora_alpha=8, target_modules=["q_proj", "v_proj"], task_type="CAUSAL_LM"))\n        for name, parameter in model.named_parameters():\n            if "lora_B" in name:\n                torch.nn.init.normal_(parameter, std=0.01)\n        with tempfile.TemporaryDirectory() as folder:\n            start, finish = Path(folder) / "start", Path(folder) / "finish"\n            model.save_pretrained(start)\n            fresh = Qwen2ForCausalLM(config)\n            fresh.load_state_dict(state)\n            warm = load_trainable_adapter(fresh, start)\n            ids = torch.randint(0, 101, (1, 16))\n            labels = ids.clone()\n            labels[:, :12] = -100\n            batch = {"input_ids": ids, "attention_mask": torch.ones_like(ids), "labels": labels}\n            loss = assistant_only_loss(warm, batch)\n            self.assertTrue(torch.isfinite(loss))\n            loss.backward()\n            self.assertTrue(any(p.grad is not None and p.grad.abs().sum() > 0 for n, p in warm.named_parameters() if "lora_" in n))\n            self.assertTrue(all(not p.requires_grad and p.grad is None for n, p in warm.named_parameters() if "lora_" not in n))\n            torch.optim.AdamW([p for p in warm.parameters() if p.requires_grad], lr=1e-3).step()\n            warm.save_pretrained(finish)\n            fresh = Qwen2ForCausalLM(config)\n            fresh.load_state_dict(state)\n            reloaded = PeftModel.from_pretrained(fresh, finish).eval()\n            warm.eval()\n            with torch.no_grad():\n                torch.testing.assert_close(warm(ids).logits, reloaded(ids).logits)\n\n\nif __name__ == "__main__":\n    unittest.main()\n', encoding='utf-8')


In [ ]:
p = Path('tests/test_augmentation_comparison.py')
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('import copy\nimport unittest\n\nfrom compare_augmentation_control import compare\n\n\ndef result():\n    predictions = [{"row": str(i), "class": "Example.cs", "question": f"Q{i}",\n                    "truth": "a", "prediction": "a" if i < 182 else "b"} for i in range(192)]\n    return {"metadata": {"model": "qwen", "model_commit": "pin", "seed": 42,\n                         "test_sha256": "test", "settings": {"num_train_epochs": 1},\n                         "input_sha256": {"validation": "val", "train": "original"},\n                         "training_source_rows": [1, 2], "smoke_test": False},\n            "validation": {"questions": 192, "correct": 182, "accuracy": 182/192},\n            "predictions": predictions}\n\n\nclass ContinuationComparisonTests(unittest.TestCase):\n    def test_single_random_protocol_difference_is_reported(self):\n        control, experiment = result(), result()\n        control["metadata"]["settings"]["rotations"] = 4\n        experiment["metadata"]["settings"]["rotations"] = 1\n        experiment["metadata"]["augmentation"] = {"presentations": 5151}\n        self.assertEqual(compare(control, experiment)["training_settings_differences"]["rotations"],\n                         {"control":4,"continuation":1})\n\n    def test_validation_pairing_allows_changed_training_budget(self):\n        control, experiment = result(), result()\n        experiment["metadata"].update(augmentation={"presentations": 20604})\n        experiment["metadata"]["input_sha256"]["train"] = "augmented"\n        experiment["predictions"][0]["prediction"] = "b"\n        experiment["predictions"][182]["prediction"] = "a"\n        comparison = compare(control, experiment)\n        self.assertEqual(comparison["wrong_to_correct"], 1)\n        self.assertEqual(comparison["correct_to_wrong"], 1)\n        self.assertEqual(comparison["accuracy_change_percentage_points"], 0)\n        self.assertEqual(len(comparison["paired_prediction_changes"]), 2)\n\n    def test_rejects_different_cohort_keys_or_fabricated_summary(self):\n        control, experiment = result(), result()\n        experiment["metadata"]["augmentation"] = {"presentations": 20604}\n        for target, value in [("question", "changed"), ("truth", "b"), ("row", "999")]:\n            bad = copy.deepcopy(experiment)\n            bad["predictions"][0][target] = value\n            with self.assertRaises(ValueError):\n                compare(control, bad)\n        bad = copy.deepcopy(experiment)\n        bad["validation"]["correct"] += 1\n        with self.assertRaisesRegex(ValueError, "summary"):\n            compare(control, bad)\n\n\nif __name__ == "__main__":\n    unittest.main()\n', encoding='utf-8')


In [ ]:
p = Path('tests/test_mcq_checkpoints.py')
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""Exercise real Trainer saves/loss evaluation and three within-epoch scores."""\nimport json\nfrom pathlib import Path\nimport tempfile\nimport unittest\n\nfrom mcq_checkpointing import make_checkpoint_callback, milestone_steps\n\n\nclass ThirdEpochTests(unittest.TestCase):\n    def test_milestones_include_final_update_and_handle_short_smoke(self):\n        self.assertEqual(milestone_steps(322), [108, 215, 322])\n        self.assertEqual(milestone_steps(1), [1])\n        self.assertEqual(milestone_steps(2), [1, 2])\n\n    def test_real_trainer_saves_three_adapters_and_actual_validation_losses(self):\n        import torch\n        from peft import LoraConfig, get_peft_model\n        from transformers import Qwen2Config, Qwen2ForCausalLM, Trainer, TrainingArguments\n        from finetune_qwen_mcq import assistant_only_loss\n        torch.set_num_threads(2)\n        torch.manual_seed(42)\n        model = Qwen2ForCausalLM(Qwen2Config(vocab_size=31, hidden_size=16, intermediate_size=32,\n            num_hidden_layers=1, num_attention_heads=2, num_key_value_heads=1, use_cache=False))\n        model = get_peft_model(model, LoraConfig(r=2, lora_alpha=4, target_modules=["q_proj", "v_proj"], task_type="CAUSAL_LM"))\n        rows = [{"input_ids": [1,2,3,4,5,6,7,8], "attention_mask": [1]*8,\n                 "labels": [-100]*5 + [6,7,8]} for _ in range(6)]\n        validation = [{"marker": "validation only"}]\n        calls = []\n        # Only text-generation accuracy is stubbed; losses and adapter saves use real Trainer.\n        def accuracy_stub(model_, tokenizer, supplied_rows, destination):\n            self.assertIs(supplied_rows, validation)\n            calls.append(model_.config.use_cache)\n            model_.eval()\n            model_.config.use_cache = True\n            torch.rand(1)  # Callback must isolate generation RNG from training.\n            destination.mkdir()\n            return {"accuracy": 0.5, "correct": 1, "questions": 2}, []\n        class TinyTrainer(Trainer):\n            def compute_loss(self, model_, inputs, return_outputs=False, num_items_in_batch=None):\n                return assistant_only_loss(model_, inputs, return_outputs, num_items_in_batch)\n        with tempfile.TemporaryDirectory() as folder:\n            output = Path(folder)\n            args = TrainingArguments(output_dir=str(output / "training"), num_train_epochs=1,\n                per_device_train_batch_size=1, per_device_eval_batch_size=1, gradient_accumulation_steps=1,\n                learning_rate=1e-4, optim="adamw_torch", eval_strategy="no", save_strategy="no",\n                save_only_model=True, report_to="none", use_cpu=True, disable_tqdm=True, logging_steps=1,\n                remove_unused_columns=False, prediction_loss_only=True, seed=42)\n            callback = make_checkpoint_callback(validation, None, output, 6, evaluate_fn=accuracy_stub)\n            trainer = TinyTrainer(model=model, args=args, train_dataset=rows, eval_dataset=rows[:2], callbacks=[callback])\n            trainer.model_accepts_loss_kwargs = True\n            trainer.train()\n            scores = json.loads((output / "checkpoint_scores.json").read_text())\n            self.assertEqual([s["optimizer_step"] for s in scores], [2,4,6])\n            self.assertEqual(scores[-1]["epoch"], 1.0)\n            self.assertEqual(calls, [False,False,False])\n            self.assertFalse(model.config.use_cache)\n            for score in scores:\n                self.assertTrue(torch.isfinite(torch.tensor(score["validation_loss"])))\n                checkpoint = output / score["checkpoint"]\n                self.assertTrue((checkpoint / "adapter_model.safetensors").is_file())\n                self.assertFalse((checkpoint / "optimizer.pt").exists())\n            self.assertEqual(len(list((output / "training").glob("checkpoint-*"))), 3)\n\n\nif __name__ == "__main__":\n    unittest.main()\n', encoding='utf-8')


In [ ]:
subprocess.run([sys.executable, '-m', 'unittest', 'tests.test_mcq_augmentation',
                'tests.test_mcq_rotation', 'tests.test_mcq_loss', 'tests.test_mcq_baseline',
                'tests.test_augmentation_comparison', 'tests.test_mcq_checkpoints', '-v'], check=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Enable GPU')
print('GPU:', torch.cuda.get_device_name(0))
for name in ['benchmark.py', 'finetune_qwen_coder.py', 'finetune_qwen_mcq.py', 'mcq_augmentation.py', 'mcq_checkpointing.py', '.tools/build_kaggle_augmentation_bundle.py', 'compare_alignment_control.py', 'compare_augmentation_control.py', 'tests/__init__.py', 'tests/test_mcq_rotation.py', 'tests/test_mcq_loss.py', 'tests/test_mcq_baseline.py', 'tests/test_mcq_augmentation.py', 'tests/test_augmentation_comparison.py', 'tests/test_mcq_checkpoints.py']:
    target = RESULT / 'scripts' / name
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(name, target)
for name in ['augmentation_manifest.json', 'control_results.json']:
    shutil.copyfile(name, RESULT / name)
if (WORK / 'generation_provenance').is_dir():
    shutil.copytree(WORK / 'generation_provenance', RESULT / 'generation_provenance')
for source in WORK.glob('control_*predictions.csv'):
    shutil.copyfile(source, RESULT / source.name)
for source in WORK.glob('control_*summary.json'):
    shutil.copyfile(source, RESULT / source.name)
def archive_results():
    return shutil.make_archive(str(RESULT), 'zip', RESULT)
def run(name, *flags):
    command = [sys.executable, '-u', 'finetune_qwen_mcq.py', '--train', 'train.csv',
               '--validation', 'validation.csv', '--test', 'TestData.csv', '--output', str(RESULT / name),
               '--fresh-base', '--augmentation-manifest', 'augmentation_manifest.json',
               '--revision', REVISION, '--skip-base-evaluation', *flags]
    try:
        with (RESULT / (name + '-console.log')).open('w', encoding='utf-8') as log:
            process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line)
                log.flush()
            if process.wait():
                raise RuntimeError(f'{name} failed; partial results archived')
    finally:
        archive_results()
run('preflight', '--prepare-only')
preflight = json.loads((RESULT / 'preflight/metadata.json').read_text())
assert preflight['training_presentations'] == MANIFEST['presentations']
assert preflight['train_label_counts'] == MANIFEST['train_label_counts']
assert preflight['training_presentation_mode'] == MANIFEST['training_presentation_mode']
assert len(preflight['training_source_rows']) == 1717
print('Actual token and step budget:', preflight['budget'])


## Fresh-base GPU smoke, then one full epoch

The full run loads original pretrained Qwen weights with a fresh LoRA adapter and optimizer. Smoke weights are discarded.


In [ ]:
import math
run('smoke', '--smoke-test')
smoke = RESULT / 'smoke'
audit = json.loads((smoke / 'warm_start_gradient_check.json').read_text())
assert audit['lora_gradients_present_finite_nonzero'] and audit['frozen_base_has_no_gradients']
assert json.loads((smoke / 'metadata.json').read_text())['fresh_lora_zero_B_verified']
assert json.loads((smoke / 'comparison.json').read_text())['saved_adapter_reload_verified']
logs = json.loads((smoke / 'training_log.json').read_text())
assert any('loss' in row for row in logs)
assert all(math.isfinite(row[key]) for row in logs for key in ['loss', 'eval_loss'] if key in row)
print('GPU smoke passed. Starting independently from pretrained Qwen with fresh LoRA.')
run('fresh_training')
if MANIFEST.get('checkpoint_evaluations'):
    scores = json.loads((RESULT / 'fresh_training/checkpoint_scores.json').read_text())
    assert len(scores) == 3
    for score in scores:
        print(f"Checkpoint {score['checkpoint_index']}: epoch={score['epoch']:.3f}, validation loss={score['validation_loss']:.4f}, accuracy={score['validation_accuracy']:.2%}")


In [ ]:
from compare_alignment_control import read_result
from compare_augmentation_control import compare
comparison = compare(read_result('control_results.json'), read_result(RESULT / 'fresh_training'))
comparison['note'] = 'Fresh pretrained Qwen plus augmented one-epoch training, compared with historical original-MCQ control. No previous adapter was loaded.'
(RESULT / 'validation_comparison.json').write_text(json.dumps(comparison, indent=2) + '\n')
print(json.dumps(comparison, indent=2))
import matplotlib.pyplot as plt
logs = json.loads((RESULT / 'fresh_training/training_log.json').read_text())
losses = [r for r in logs if 'loss' in r]
plt.plot([r['step'] for r in losses], [r['loss'] for r in losses])
plt.xlabel('Optimizer step'); plt.ylabel('Answer-only training loss')
plt.savefig(RESULT / 'loss.png', bbox_inches='tight'); plt.show()
archive_results()

print('FINAL SCORES:', json.dumps(json.loads((RESULT / 'fresh_training/comparison.json').read_text())['finetuned'], indent=2))


## Interactive MCQ demo

Load the saved freshly trained adapter and use `demo(question, a, b, c, d)`. The final test and rotated validation outputs are retained under fresh_training/finetuned.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from finetune_qwen_coder import MODEL_ID
from benchmark import build_prompt, parse_answer
tokenizer = AutoTokenizer.from_pretrained(RESULT / 'fresh_training/adapter')
base = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=REVISION, device_map={'': 0},
    dtype=torch.float16, quantization_config=BitsAndBytesConfig(load_in_4bit=True,
    bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16))
model = PeftModel.from_pretrained(base, RESULT / 'fresh_training/adapter').eval()
def demo(question, a, b, c, d):
    row = dict(Question=question, a=a, b=b, c=c, d=d)
    ids = tokenizer.apply_chat_template([{'role': 'user', 'content': build_prompt(row)}],
        add_generation_prompt=True, return_tensors='pt').to(model.device)
    if ids.shape[-1] > 4096:
        raise ValueError('Demo prompt exceeds reviewed context')
    with torch.inference_mode():
        output = model.generate(ids, max_new_tokens=128, do_sample=False, pad_token_id=tokenizer.pad_token_id)
    text = tokenizer.decode(output[0, ids.shape[-1]:], skip_special_tokens=True)
    return {'response': text, 'prediction': parse_answer(text)}
import csv
with Path('validation.csv').open(encoding='utf-8-sig', newline='') as handle:
    row = next(csv.DictReader(handle))
demo_check = demo(row['Question'], row['a'], row['b'], row['c'], row['d'])
(RESULT / 'demo_check.json').write_text(json.dumps(demo_check, indent=2))
print('Reloaded demo response:', demo_check)
from IPython.display import FileLink, display
display(FileLink(archive_results()))
